# Benchmarks analysis notebook

This notebook is used to analyze the benchmarks results and to generate the plots that are used in the benchmarks report.

This report contains executions for techniques:
- TNC 
- TFC 
- Diet
- LFR 
- Supervised

and for backbones:
- CNN PF
- Resnet
- RNN
- Transformers
- TS2VEC 

## Define base configs

In [227]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy"
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [228]:
import sys
from pathlib import Path

# Use the current working directory instead of __file__
sys.path.append(str(Path.cwd().parent))

# use utils from previous folder
from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [229]:
# Path to the experiment results (parsed) summarized_executions_fixed
# filename = '../saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_harcnn_12345old'

# filename = 'saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_resnetse5_12345'
# filename = 'saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_harcnn'
filename = 'saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed'
# filename = 'saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

Sucessfully created directories 'results/figures' and 'results/tables'


## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [230]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [231]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



## Extractor functions

This should be implemented by user, as it is specific to the problem. 
With executions' dataframe in hand, the user should implement/modify the following functions, in order to return the appropriate values:

- `extract_backbone`: Should return the backbone used based on information in the executions' dataframe (e.g., TS2Vec).
- `extract_tsk_pretext`: Should return the name of the pretext task used based on information in the executions' dataframe (e.g., CPC).
- `extract_d_pretext`: Should return the name of the pretext dataset used in the execution of the pretext task (e.g., KuHAR).
- `extract_tsk_target`: Should return the name of the target task (e.g., HAR, Authentication, etc).
- `extract_ft_strategy`: Should return the name of the fine-tuning strategy used on downstream task (e.g., Freeze).
- `extract_head_pred`: Should return the name of the prediction head used on downstream task (e.g., MLP).
- `extract_d_target`: Should return the name of the target dataset used in the execution of the downstream task (e.g., KuHAR).
- `extract_frac_dtarget`: Should return the fraction of the target dataset used in the execution of the downstream task (e.g., 0.1).
- `extract_metric_target`: Should return the metric used to evaluate the performance of the model on the target task (e.g., Accuracy).
- `extract_metric` should return the value of the metric used to evaluate the performance of the model on the target task, based on desired metric (e.g., 0.9).


**Note**: All extractor functions should return a list of values of same length as the number of executions.
**Note**: All functions receives the same dataframe as input (executions_df), so they can use it to extract the values. All functions should return a list of values, one for each execution (line in the dataframe).

Users should change the implementation of the following functions to match the problem at hand. Once the functions are implemented, a dataframe with the extracted values will be created. This dataframe will be used in the rest of the notebook to generate the plots and tables. Besides that, the dataframe follows the same structure as described in the H.IAAC evaluation methodology report.

In [232]:
def extract_backbone(df):

    backbone_map = {
        'tfc_cnnpff':"CNN PF", 
        'rnn': "RNN",
        'resnet':"ResNet-1D",
        'tnc_resnet':"ResNet-1D",
        'cnnpff':"CNN PF",
       'tnc_cnnpff':"CNN PF", 
       'tfc_resnet':"ResNet-1D",
       'transformer':"IMU Transformer",
        'tfc_transformer':"IMU Transformer",
       'tfc_rnn': "RNN",
        'tnc_rnn': "RNN",
        'tnc_transformer':"IMU Transformer",
        'diet_rnn': "RNN",
        'diet_rnn_1': "RNN",
        'diet_rnn_2': "RNN",
        'diet_rnn_7': "RNN",
        'diet_transformer_1': "IMU Transformer",
        'diet_transformer_2': "IMU Transformer",
        'diet_transformer_7': "IMU Transformer",
        'diet_transformer': "IMU Transformer",
        'diet_cnnpff': "CNN PF",
        'diet_cnnpff_1': "CNN PF",
        'diet_cnnpff_2': "CNN PF",
        'diet_cnnpff_7': "CNN PF",
        'diet_resnet': "ResNet-1D",
        'diet_resnet_1': "ResNet-1D",
        'diet_resnet_2': "ResNet-1D",
        'diet_resnet_7': "ResNet-1D",
        'lfr_resnet_1': "ResNet-1D",
        'lfr_resnet_2': "ResNet-1D",
        'lfr_resnet_7': "ResNet-1D",
        'lfr_transformer_1': "IMU Transformer",
        'lfr_transformer_2': "IMU Transformer",
        'lfr_transformer_7': "IMU Transformer",
        'lfr_rnn_1': "RNN",
        'lfr_rnn_2': "RNN",
        'lfr_rnn_7': "RNN",
        'lfr_cnnpff_1': "CNN PF",
        'lfr_cnnpff_2': "CNN PF",
        'lfr_cnnpff_7': "CNN PF",
        'lfr_resnet': "ResNet-1D",
        'lfr_transformer': "IMU Transformer",
        'lfr_rnn': "RNN",
        'lfr_cnnpff': "CNN PF",
        'ts2vec': "TS Encoder",
        'tnc_ts2vec': "TS Encoder",
        'tfc_ts2vec': "TS Encoder",
        'diet_ts2vec': "TS Encoder",
        'diet_ts2vec_1': "TS Encoder",
        'lfr_ts2vec': "TS Encoder",
        'harcnn': "HAR CNN",
        'resnetse5': "ResNet-SE5",

    }


    return df["model/name"].map(backbone_map)



def extract_tsk_pretext(df):
    tsk_pretext_map = {
        'tfc_cnnpff':"TFC", 
        'rnn': "Supervised",
        'resnet':"Supervised",
        'tnc_resnet':"TNC",
        'cnnpff':"Supervised",
       'tnc_cnnpff':"TNC", 
       'tfc_resnet':"TFC",
       'transformer':"Supervised",
        'tfc_transformer':"TFC",
       'tfc_rnn': "TFC",
        'tnc_rnn': "TNC",
        'tnc_transformer':"TNC",
        'diet_rnn': "Diet",
        'diet_transformer': "Diet",
        'diet_cnnpff': "Diet",
        'diet_resnet': "Diet",
        'diet_rnn_1': "Diet",
        'diet_transformer_1': "Diet",
        'diet_cnnpff_1': "Diet",
        'diet_resnet_1': "Diet",
        'lfr_rnn': "LFR",
        'lfr_transformer': "LFR",
        'lfr_cnnpff': "LFR",
        'lfr_resnet': "LFR",
        'lfr_rnn_1': "LFR",
        'lfr_transformer_1': "LFR",
        'lfr_cnnpff_1': "LFR",
        'lfr_resnet_1': "LFR",
        'lfr_rnn_2': "LFR",
        'lfr_transformer_2': "LFR",
        'lfr_cnnpff_2': "LFR",
        'lfr_resnet_2': "LFR",
        'lfr_rnn_7': "LFR",
        'lfr_transformer_7': "LFR",
        'lfr_cnnpff_7': "LFR",
        'lfr_resnet_7': "LFR",
        'diet_rnn_2': "Diet",
        'diet_transformer_2': "Diet",
        'diet_cnnpff_2': "Diet",
        'diet_resnet_2': "Diet",
        'diet_rnn_7': "Diet",
        'diet_transformer_7': "Diet",
        'diet_cnnpff_7': "Diet",
        'diet_resnet_7': "Diet",
        'ts2vec': "Supervised",
        'tnc_ts2vec': "TNC",
        'tfc_ts2vec': "TFC",
        'diet_ts2vec': "Diet",
        'diet_ts2vec_1': "Diet",
        'lfr_ts2vec': "LFR",
        'harcnn': "Supervised",
        'resnetse5': "Supervised",

        
    }
    return df["model/name"].map(tsk_pretext_map)


def extract_d_pretext(df):
    d_pretext_map = {
        "kuhar": "KH",
        "motionsense": "MS",
        "rw_thigh": "RW-Thigh",
        "rw_waist": "RW-Waist",
        "uci": "UCI",
        "wisdm": "WISDM",
        "hapt": "HAPT",
        "recodgait": "RecodGait",
    }
    return df["data/dataset"].map(d_pretext_map)


def extract_ft_strategy(df):
    def extract_ft_by_name(row):
        if "freeze" in row["model/override_id"]:
            return "Freeze"
        else:
            return "Full Finetune"
    
    return df.apply(extract_ft_by_name, axis=1)

def extract_head_pred(df):
    return ["MLP"] * len(df)


def extract_tsk_target(df):
    tsk_map = {
        "har": "HAR",
    }
    return df["pipeline/task"].map(tsk_map)


def extract_d_target(df):
    d_target_map = {
        "kuhar": "KH",
        "motionsense": "MS",
        "rw_thigh": "RW-Thigh",
        "rw_waist": "RW-Waist",
        "uci": "UCI",
        "wisdm": "WISDM",
        "hapt": "HAPT",
    }
    return df["data/dataset"].map(d_target_map)



def extract_frac_dtarget(df):
    def get_mix_percentage(row):
        frac_dtarget_map = {
            "multimodal_samples_001": 1,
            "multimodal_samples_001_2": 1,
            "multimodal_samples_001_3": 1,
            "multimodal_samples_005": 5,
            "multimodal_samples_005_2": 5,
            "multimodal_samples_005_3": 5,
            "multimodal_samples_010": 10,
            "multimodal_samples_010_2": 10,
            "multimodal_samples_010_3": 10,
            "multimodal_samples_025": 25,
            "multimodal_samples_025_2": 25,
            "multimodal_samples_025_3": 25,
            "multimodal_samples_050": 50,
            "multimodal_samples_050_2": 50,
            "multimodal_samples_050_3": 50,
            "multimodal_samples_100": 100,
            "multimodal_samples_100_2": 100,
            "multimodal_samples_100_3": 100,
            "multimodal_samples_200": 200,
            "multimodal_samples_200_2": 200,
            "multimodal_samples_200_3": 200,
            "multimodal_perc_100": 1000,
            }
        if not pd.isna(row["backbone/load_from_uid"] ):
            try:
                row = df.loc[df["execution/uid"] == row["backbone/load_from_uid"]].iloc[0]

                frac = row["data/override_id"]
                return frac_dtarget_map[frac]
            except Exception as e:
                print(f"Invalid load_from_uid: {row['backbone/load_from_uid']}:\n {e}")
                return 0
        
    return df.apply(get_mix_percentage, axis=1)


In [233]:
def aggregate_results(df: pd.DataFrame) -> pd.DataFrame:
    """Group the dataframe by all columns except the metric and aggregate the
    metric values using the mean and standard deviation.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be aggregated

    Returns
    -------
    _type_
        _description_
    """
    agg_df = []
    every_column_expect_metric = [col for col in df.columns if col != "metric"]

    for _, grouped_df in df.groupby(every_column_expect_metric):
        grouped_df = grouped_df.copy()
        mean = grouped_df["metric"].mean()
        stdev = grouped_df["metric"].std()
        if pd.isna(stdev):
            stdev = 0.0
        grouped_df["metric"] = mean
        grouped_df["metric_stdev"] = stdev
        grouped_df["run_count"] = len(grouped_df)

        # print(f"{group_name}, with values: {grouped_df['metric'].values} -- {mean:.2f} ± {stdev:.2f}")
        single_line = grouped_df.iloc[0]
        agg_df.append(single_line)

    df = pd.DataFrame(agg_df).reset_index(drop=True)
    return df


def parse_dataframe(
    df: pd.DataFrame,
    filter_nan_metric: bool = True,
    aggregate_runs: bool = False,
) -> pd.DataFrame:
    """Parse the dataframe to extract the relevant columns and values for the
    experiment. If `filter_nan_metric` is True, the rows with NaN metric values
    are removed. If `aggregate_runs` is True, the results are aggregated by
    grouping the dataframe by all columns except the metric and calculating the
    mean and standard deviation of the metric values.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be parsed
    filter_nan_metric : bool, optional
        If True the rows with NaN metric values are removed, by default True
    aggregate_runs : bool, optional
        If True the results are aggregated, by default False

    Returns
    -------
    pd.DataFrame
        The parsed dataframe
    """
    d = {
        # "uid": pd.Series(df["execution/uid"], dtype="object"),
        "backbone": pd.Series(extract_backbone(df), dtype="object"),
        "tsk_pretext": pd.Series(extract_tsk_pretext(df), dtype="object"),
        "d_pretext": pd.Series(extract_d_pretext(df), dtype="object"),
        "head_pred": pd.Series(extract_head_pred(df), dtype="object"),
        "ft_strategy": pd.Series(extract_ft_strategy(df), dtype="object"),
        "tsk_target": pd.Series(extract_tsk_target(df), dtype="object"),
        "d_target": pd.Series(extract_d_target(df), dtype="object"),
        "frac_dtarget": pd.Series(extract_frac_dtarget(df), dtype="float"),
        "metric_target": pd.Series(extract_metric_target(df), dtype="object"),
        "metric": pd.Series(extract_metric(df), dtype="float"),
        "lr": pd.Series(df["model/override_id"], dtype="float"),
    }

    df = pd.DataFrame(d)

    if filter_nan_metric:
        df = df.dropna(subset=["metric"])

    if aggregate_runs:
        df = aggregate_results(df)
    return df

In [234]:
df = pd.read_csv(summarized_executions_path)
df

,execution/id,execution/uid,backbone/load_from_uid,execution/status,execution/num_deps,ckpt/resume,model/uid,model/config,model/name,model/override_id,...,execution/root_dir,execution/run_id,execution/metric_file,execution/run_file,metric/classification/accuracy,metric/classification/balanced_accuracy,metric/classification/f1_score_macro,metric/classification/f1_score_micro,metric/classification/precision,metric/classification/recall
0,train_cnnpff,id_000d95f83256,NaN,completed,0,False,id_4e03b495b990,supervised,cnnpff,full_finetune_lr2,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,train_resnet,id_000f27360cbc,NaN,completed,0,False,id_e4f9858e2aed,supervised,resnet,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train_rnn,id_00258c4c73f3,NaN,completed,0,False,id_13f3a0ef598f,supervised,rnn,full_finetune_lr3,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,train_cnnpff,id_001d64dac16a,NaN,completed,0,False,id_18ba8cda9379,supervised,cnnpff,full_finetune_lr5,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,train_transformer,id_004393aa92bd,NaN,completed,0,False,id_b8d0f0b27e64,supervised,transformer,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7195,evaluate_ts2vec,id_e6f4623f5442,id_53e80d2a1436,completed,1,True,id_23c273ffc330,supervised,ts2vec,full_finetune_lr5,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,4140962c8f,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.250000,0.250000,0.100000,0.250000,0.250000,0.062500
7196,evaluate_ts2vec,id_f94f8238c62a,id_61e97db9686c,completed,1,True,id_23c273ffc330,supervised,ts2vec,full_finetune_lr5,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,26e576bcd6,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.250000,0.250000,0.100000,0.250000,0.250000,0.062500
7197,evaluate_ts2vec,id_f670fb59d8bc,id_8c33c93835dd,completed,1,True,id_23c273ffc330,supervised,ts2vec,full_finetune_lr5,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,59a52e2fa6,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.167357,0.583391,0.049022,0.167357,0.167357,0.194464
7198,evaluate_ts2vec,id_fa5f6ea2780f,id_33f425fdf7d0,completed,1,True,id_23c273ffc330,supervised,ts2vec,full_finetune_lr5,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,3179b66c9c,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.250000,0.250000,0.100000,0.250000,0.250000,0.062500


In [235]:
df['model/config'].unique()

array(['supervised'], dtype=object)

In [236]:
df['model/override_id'].unique()

array(['full_finetune_lr2', 'full_finetune_lr4', 'full_finetune_lr3',
       'full_finetune_lr5', 'full_finetune_lr1'], dtype=object)

In [237]:
# # Filter rows where 'model/override_id' contains 'lr3' or 'lr5'
# df_filtered = df[~df['model/override_id'].str.contains('lr3|lr5', na=False)]


df_filtered = df[df['model/override_id'].str.contains('lr4', na=False)]
# df_filtered = df[~df['model/config'].str.contains('supervised', na=False)]
# df_filtered.to_csv("saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_resnetse5_4.csv", index=False)
# # Save the filtered dataframe if needed
# df_filtered.to_csv("saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_harcnn_124old.csv", index=False)

In [238]:
df_filtered

,execution/id,execution/uid,backbone/load_from_uid,execution/status,execution/num_deps,ckpt/resume,model/uid,model/config,model/name,model/override_id,...,execution/root_dir,execution/run_id,execution/metric_file,execution/run_file,metric/classification/accuracy,metric/classification/balanced_accuracy,metric/classification/f1_score_macro,metric/classification/f1_score_micro,metric/classification/precision,metric/classification/recall
1,train_resnet,id_000f27360cbc,NaN,completed,0,False,id_e4f9858e2aed,supervised,resnet,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,train_transformer,id_004393aa92bd,NaN,completed,0,False,id_b8d0f0b27e64,supervised,transformer,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,train_resnet,id_00a378570ce3,NaN,completed,0,False,id_e4f9858e2aed,supervised,resnet,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,train_resnet,id_00e9eb994ff6,NaN,completed,0,False,id_e4f9858e2aed,supervised,resnet,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,train_cnnpff,id_01f0c8f807d6,NaN,completed,0,False,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6322,evaluate_ts2vec,id_f12463e014d5,id_6901b035aef6,completed,1,True,id_fe14fa6bfd69,supervised,ts2vec,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0291b575b6,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.761960,0.785446,0.762763,0.761960,0.761960,0.785446
6323,evaluate_ts2vec,id_f533c1a8fe51,id_47bf11d33278,completed,1,True,id_fe14fa6bfd69,supervised,ts2vec,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,2575f1e088,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.805797,0.812208,0.794909,0.805797,0.805797,0.812208
6325,evaluate_ts2vec,id_f23a263513c4,id_6a28b3b3fde9,completed,1,True,id_fe14fa6bfd69,supervised,ts2vec,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,489d47c4f7,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.632104,0.681714,0.548039,0.632104,0.632104,0.511285
6327,evaluate_ts2vec,id_f2e629762def,id_8edfe9a552d1,completed,1,True,id_fe14fa6bfd69,supervised,ts2vec,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,5958e378a0,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.875576,0.875390,0.875437,0.875576,0.875576,0.875390


In [239]:
df_filtered = df_filtered[df_filtered['execution/id'].str.contains('evaluate_', na=False)]

df_filtered = df_filtered[df_filtered['execution/id'].str.contains('cnn', na=False)]

In [240]:
df_filtered

,execution/id,execution/uid,backbone/load_from_uid,execution/status,execution/num_deps,ckpt/resume,model/uid,model/config,model/name,model/override_id,...,execution/root_dir,execution/run_id,execution/metric_file,execution/run_file,metric/classification/accuracy,metric/classification/balanced_accuracy,metric/classification/f1_score_macro,metric/classification/f1_score_micro,metric/classification/precision,metric/classification/recall
2886,evaluate_cnnpff,id_00a9b348c1fb,id_1fe43781bee8,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,125bc8181a,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.711870,0.736233,0.713510,0.711870,0.711870,0.736233
2919,evaluate_cnnpff,id_0b899167f5b8,id_886aa20c9d85,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,5515a6f9d3,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.687371,0.745344,0.678253,0.687371,0.687371,0.745344
2929,evaluate_cnnpff,id_0cf1d1d284dc,id_8710e9239c4f,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,58dbc1989b,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.585262,0.646260,0.525991,0.585262,0.585262,0.538550
2933,evaluate_cnnpff,id_0d3a799e8d12,id_93bcd879fb58,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,1460fce463,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.569124,0.502202,0.505487,0.569124,0.569124,0.502202
2939,evaluate_cnnpff,id_0e0498a48bc2,id_82ce62dde775,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,389cf5e60c,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.555556,0.582963,0.554676,0.555556,0.555556,0.582963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4923,evaluate_cnnpff,id_f9a5b3d2e1c1,id_40f6e64798c5,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,21a9fbf269,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.548611,0.715758,0.429356,0.548611,0.548611,0.596465
4926,evaluate_cnnpff,id_fa05d6211247,id_629f610e25ec,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,2626b7b4a1,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.813559,0.863241,0.795865,0.813559,0.813559,0.863241
4945,evaluate_cnnpff,id_fbc57b20df65,id_80503c9d8a88,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,2724c7d440,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.642857,0.668038,0.556333,0.642857,0.642857,0.668038
4973,evaluate_cnnpff,id_fefc0f5ce0d4,id_6e11f4c757cd,completed,1,True,id_f7b27f562c75,supervised,cnnpff,full_finetune_lr4,...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,43221118c0,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,/workspaces/HIAAC-KR-Dev-Container/Minerva-Exp...,0.791304,0.806107,0.761212,0.791304,0.791304,0.806107


In [241]:
# mean of metric/classification/accuracy
df_filtered['metric/classification/accuracy'].mean().round(3)

0.634

In [242]:
df_filtered['metric/classification/accuracy'].std()

0.18811261993605224

In [243]:
df_filtered['model/config'].unique()

array(['supervised'], dtype=object)

In [244]:
# # Filter rows where 'model/override_id' contains 'lr3' or 'lr5'
# df_filtered = df[~df['model/override_id'].str.contains('lr3|lr5', na=False)]

# # Save the filtered dataframe if needed
# df_filtered.to_csv("saved_metrics_paper_daghar_supervised_3runs_perlr_samples_down_seed_harcnn_124old.csv", index=False)

In [245]:
a

NameError: name 'a' is not defined

In [ ]:
# start_idx = 7500
# end_idx = 8500  # exclusive

# # Step 1: Split the DataFrame into three parts
# before = df.iloc[:start_idx]
# middle = df.iloc[start_idx:end_idx]
# after = df.iloc[end_idx:]

# # Step 2: Filter out rows containing 'tnc_resnet' in the middle part
# # You can specify the column or check across all columns (assuming all string columns)
# # Here's how to check all columns for 'tnc_resnet' string:
# mask = ~middle.astype(str).apply(lambda row: row.str.contains('tnc_resnet')).any(axis=1)
# middle_filtered = middle[mask]

# filter = False

# # Step 3: Concatenate back
# if filter:
#     df = pd.concat([before, middle_filtered, after], ignore_index=True)

# df

In [ ]:
df['model/name'].unique()

In [ ]:
df = parse_dataframe(df, filter_nan_metric=True, aggregate_runs=False)
df

In [ ]:
df

In [ ]:
df['lr'] = df.iloc[:, -1].str.extract(r'lr(\d+)', expand=False).astype(str)
df


In [ ]:
df['lr'] = '1e-' + df['lr']
df


In [ ]:
df['tsk_pretext'] = df['lr']
df

In [ ]:
df["frac_dtarget"] = df["frac_dtarget"].astype(str)
# df = df[~df['frac_dtarget'].str.contains('100')]
df

In [ ]:
# df = df[~df['frac_dtarget'].str.contains('50')]
df

In [ ]:
df = df[~df['d_pretext'].str.contains('HAPT')]

In [ ]:
df

In [ ]:
df.info()

In [ ]:
from tabulate import tabulate  # optional, for prettier markdown

columns = ['tsk_pretext', 'backbone', 'd_target', 'frac_dtarget', 'ft_strategy']

# Prepare the rows
rows = []
for col in columns:
    counts = df[col].value_counts()
    summary = ", ".join([f"{k} ({v})" for k, v in counts.items()])
    rows.append((col, summary))

# Print Markdown Table
print(f'number of experiments: {len(df)}')
print("## 📊 Markdown Summary Table\n")
print(tabulate(rows, headers=["Variable", "Value Counts"], tablefmt="github"))

# Build LaTeX Table
latex_table = "\\begin{table}[ht]\n\\centering\n"
latex_table += "\\begin{tabular}{ll}\n"
latex_table += "\\toprule\n"
latex_table += "Variable & Value Counts \\\\\n"
latex_table += "\\midrule\n"
for var, vals in rows:
    latex_table += f"{var} & {vals} \\\\\n"
latex_table += "\\bottomrule\n"
latex_table += "\\end{tabular}\n"
latex_table += "\\caption{Summary of experimental settings used across different configurations.}\n"
latex_table += "\\label{tab:exp-summary}\n"
latex_table += "\\end{table}"

print("\n\n## 📄 LaTeX Table\n")
print(latex_table)


In [ ]:
# convert fracdtarget 0.00 to 1000
df["frac_dtarget"] = df["frac_dtarget"].replace("0.0", "1000.0")

In [ ]:
from tabulate import tabulate  # optional, for prettier markdown

columns = ['tsk_pretext', 'backbone', 'd_target', 'frac_dtarget', 'ft_strategy']

# Prepare the rows
rows = []
for col in columns:
    counts = df[col].value_counts()
    summary = ", ".join([f"{k} ({v})" for k, v in counts.items()])
    rows.append((col, summary))

# Print Markdown Table
print(f'number of experiments: {len(df)}')
print("## 📊 Markdown Summary Table\n")
print(tabulate(rows, headers=["Variable", "Value Counts"], tablefmt="github"))

# Build LaTeX Table
latex_table = "\\begin{table}[ht]\n\\centering\n"
latex_table += "\\begin{tabular}{ll}\n"
latex_table += "\\toprule\n"
latex_table += "Variable & Value Counts \\\\\n"
latex_table += "\\midrule\n"
for var, vals in rows:
    latex_table += f"{var} & {vals} \\\\\n"
latex_table += "\\bottomrule\n"
latex_table += "\\end{tabular}\n"
latex_table += "\\caption{Summary of experimental settings used across different configurations.}\n"
latex_table += "\\label{tab:exp-summary}\n"
latex_table += "\\end{table}"

print("\n\n## 📄 LaTeX Table\n")
print(latex_table)


In [ ]:
df.to_csv(f'clean_{filename}.csv', index=False)

In [ ]:
base_df = df.copy()

In [ ]:
df

In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        "select_tsk_pretext": ['1e-3','1e-4','1e-2','1e-1','1e-5'],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        "select_backbones": ["HAR CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        # "select_tsk_pretext": [ "S","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_ssl_rq1"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

summary_df_ssl_rq12




In [ ]:
import numpy as np

for backbone in df['backbone'].unique():



    # SSL without ts2vec - will be definitive
    dot,df_ssl_rq12 = show_precedence_graph(
        df,
        variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
        filters={
            # "select_tsk_pretext": ['1e-3','1e-4'],               # Only use the full dataset
            # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
            "select_backbones": [backbone],   # "TS2Vec",  Only use these backbones "TS2Vec",
            # "select_tsk_pretext": [ "S","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
        },
        show_stdev=True,
        apply_bonferroni= apply_correction_factor,
        filename = "wilcoxon_ssl_rq1"
        
    )




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
# summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

# summary_df_ssl_rq12




In [ ]:
df

In [ ]:
import numpy as np
dfs_list = []
for backbone in df['backbone'].unique():



    # SSL without ts2vec - will be definitive
    dot,df_ssl_rq12 = show_precedence_graph(
        df,
        variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
        filters={
            # "select_tsk_pretext": ['1e-3','1e-4'],               # Only use the full dataset
            # "select_frac_dtarget": ["200.0","1000.0",],    # Only use the full finetuning strategy
            "select_backbones": [backbone],   # "TS2Vec",  Only use these backbones "TS2Vec",
            # "select_tsk_pretext": [ "S","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
        },
        show_stdev=True,
        apply_bonferroni= apply_correction_factor,
        filename = "wilcoxon_ssl_rq1"
        
    )




    summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
    display(summary_df_ssl_rq12)
    dfs_list.append(summary_df_ssl_rq12)

summary_df_ssl_rq123 = pd.concat(dfs_list, ignore_index=True)
display(summary_df_ssl_rq123)
# summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

# summary_df_ssl_rq12




In [ ]:
summary_df_ssl_rq12 = summary_df_ssl_rq123.copy()

import pandas as pd
from tabulate import tabulate


# df = pd.DataFrame(data)

# Sort by backbone and then by learning rate for better organization
summary_df_ssl_rq12[['Backbone', 'LR']] = summary_df_ssl_rq12['Backbone'].str.split('+', expand=True)
summary_df_ssl_rq12['LR'] = summary_df_ssl_rq12['LR'].str.extract(r'(\d+e-\d+)')[0]
# summary_df_ssl_rq12['LR_num'] = summary_df_ssl_rq12['LR'].str.replace('e-', '').astype(int)
summary_df_ssl_rq12 = summary_df_ssl_rq12.sort_values(['Backbone', 'LR'])
# summary_df_ssl_rq12 = summary_df_ssl_rq12.drop(['LR', 'LR_num'], axis=1)
display(summary_df_ssl_rq12)
# Format for LaTeX table
latex_table = tabulate(summary_df_ssl_rq12, headers='keys', tablefmt='latex_raw', showindex=False)

# Format for markdown (for documentation)
markdown_table = tabulate(summary_df_ssl_rq12, headers='keys', tablefmt='github', showindex=False)

print("LaTeX Table:")
print(latex_table)
print("\nMarkdown Table:")
print(markdown_table)

In [ ]:

display(summary_df_ssl_rq12)

In [ ]:
a

In [ ]:
# tfc cnn ts2vec across datasets and fractions
import numpy as np
# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["1e-1"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "1e-1",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# supervised cnn ts2vec across datasets and fractions

# with ts2vec partial

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["1e-2"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "1e-2",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["1e-3"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "1e-3",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["1e-4"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "1e-4",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["1e-5"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "1e-5",
]

# Generate all tables
combined_df, technique_summary_1e55, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_1e55)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
combined_df = pd.concat([
    # technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_diet,
    technique_summary_tnc,
    technique_summary_1e55
])


combined_df = combined_df.drop_duplicates(subset=['Backbone','Technique'])
combined_df

In [ ]:



combined_df['Data Percentage'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[2].str.replace('.0', '').astype(int)
combined_df['Dataset Name'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[1]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df['Backbone'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[0]


combined_df

In [ ]:
combined_df

In [ ]:
combined_df['Technique'] = pd.Categorical(
    combined_df['Technique'],
    categories=['1e-1', '1e-2', '1e-3', '1e-4','1e-5'],  # your desired order
    ordered=True
)
combined_df

In [ ]:
combined_df

In [ ]:

import plotly.express as px

fig = px.box(
    combined_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    points="all",
    hover_data=['Dataset Name', 'Data Percentage', 'Std'],
    title='Performance by Backbone and Technique At Full Finetuning',
    color_discrete_map={
        'Supervised': '#636EFA',  # Plotly Blue
        'TNC': '#EF553B',         # Crimson Red
        'LFR': '#00CC96',         # Green
        'TFC': '#FF8C00' ,         # Orange
        'Diet': '#9467BD', # Purple 
         
    }
)
# fig.update_layout(xaxis={'categoryorder':'total descending'})

fig.update_layout(
    title_font=dict(size=30),
    font=dict(size=18),  # General font size (tick labels, legend, etc.)
    legend=dict(font=dict(size=14)),
    xaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        # categoryorder=''
    ),
    yaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        range=[0, 100]  # ⬅️ Force y-axis from 0 to 100
    )
)
fig.show()

fig.write_html("experiment_lr3_down_samples_seed_freeze_boxplot.html")
fig.write_image("experiment_lr3_down_samples_seed_freeze__boxplot.png", width=1200, height=800)



In [ ]:
combined_df['ft_strategy'] = 'Freeze'
combined_df

In [ ]:
combined_df_freeze = combined_df.copy()

In [ ]:
# Group by technique and find worst mean performance
worst_by_technique = combined_df.groupby('Technique')['Mean'].agg(['min', 'idxmin'])
worst_technique_results = combined_df.loc[worst_by_technique['idxmin']].sort_values('Mean')

display(worst_technique_results[['Technique', 'Dataset Name', 'Data Percentage', 'Backbone', 'Mean', 'Std']])

In [ ]:
# Group by data percentage and find worst mean performance
worst_by_percentage = combined_df.groupby('Data Percentage')['Mean'].agg(['min', 'idxmin'])
worst_percentage_results = combined_df.loc[worst_by_percentage['idxmin']].sort_values('Data Percentage')

display(worst_percentage_results[['Data Percentage', 'Dataset Name', 'Technique', 'Backbone', 'Mean', 'Std']])

In [ ]:
combined_df_ft = combined_df.copy()
combined_df_ft['ft_strategy'] = 'Full Finetune'

In [ ]:
# combined_df_all = pd.concat([combined_df_ft, combined_df_freeze])
combined_df_all = combined_df_ft

In [ ]:
combined_df_all.rename(columns={
    'Data Percentage': 'Samples per Class',
}, inplace=True)
combined_df_all

In [ ]:
import plotly.express as px

fig = px.box(
    combined_df_all,
    x='Backbone',
    y='Mean',
    color='Technique',
    points="all",
    hover_data=['Dataset Name', 'Samples per Class', 'ft_strategy', 'Std'],
    title='Performance Distribution by Backbone and Technique',
    color_discrete_map={
        'Supervised': '#636EFA',  # Plotly Blue
        'TNC': '#EF553B',         # Crimson Red
        'LFR': '#00CC96',         # Green
        'TFC': '#FF8C00' ,         # Orange
        'Diet': '#9467BD', # Purple 
         
    },
    # facet_col='ft_strategy',
)
# fig.update_layout(xaxis={'categoryorder':'total descending'})

fig.update_layout(
    title_font=dict(size=30),
    font=dict(size=18),  # General font size (tick labels, legend, etc.)
    legend=dict(font=dict(size=14)),
    xaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        # categoryorder='total descending'
    ),
    yaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        range=[0, 100]  # ⬅️ Force y-axis from 0 to 100
    )
)
fig.show()

fig.write_html("experiment_lr3_down_samples_seed_boxplot.html")
fig.write_image("experiment_lr3_down_samples_seed_boxplot.png", width=1200, height=800)



In [ ]:
combined_df

In [ ]:
combined_df = combined_df[['Dataset Name','Backbone','Technique','Data Percentage','Mean','Std']]
combined_df

In [ ]:
combined_df = combined_df.rename(columns={'Dataset Name':'Dataset','Data Percentage':'Data_Percentage'})
combined_df

In [ ]:
combined_df['Mean'] = pd.to_numeric(combined_df['Mean'], errors='coerce')
combined_df['Std'] = pd.to_numeric(combined_df['Std'], errors='coerce')


In [ ]:
plot_df = combined_df.copy()
plot_df

In [ ]:
plot_df

In [ ]:
plot_df.info()

In [ ]:
plot_df['Technique'] = plot_df['Technique'].astype(str)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D
import re

# Assuming your data is in a DataFrame called 'df'
# df = pd.read_csv(...) or however you're loading your data

# 1. First find the best performance for each (dataset, data percentage, technique) combination
def get_best_performance(df):
    # Group by relevant columns and get the max mean for each group
    return df.loc[df.groupby(['Dataset', 'Data_Percentage', 'Technique', 'Backbone'])['Mean'].idxmax()]

best_perf = get_best_performance(plot_df)
best_perf

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
from matplotlib.lines import Line2D

# 1. Fixed comparison table function
def create_comparison_table(df):
    # Get best supervised for each (dataset, data percentage)
    supervised = df#[df['Technique'] == 'Supervised'].copy()
    supervised = supervised.loc[supervised.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]
    display(supervised)
    supervised['Best_Supervised'] = supervised.apply(
        lambda x: f"{x['Mean']:.1f}±{x['Std']:.1f}({x['Backbone']})" if not pd.isna(x['Std']) 
        else f"{x['Mean']:.1f}({x['Backbone']})", axis=1)
    
    # Get best SSL (all techniques except Supervised)
    ssl = df#[df['Technique'] != 'Supervised'].copy()
    ssl = ssl.loc[ssl.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]
    ssl['Best_SSL'] = ssl.apply(
        lambda x: f"{x['Mean']:.1f}±{x['Std']:.1f}({x['Backbone']},{x['Technique']})" if not pd.isna(x['Std']) 
        else f"{x['Mean']:.1f}({x['Backbone']},{x['Technique']})", axis=1)
    display(ssl)
    # Calculate SSL impact correctly
    comparison = pd.merge(
        supervised[['Dataset', 'Data_Percentage', 'Best_Supervised', 'Mean']].rename(columns={'Mean': 'Supervised_Mean'}),
        ssl[['Dataset', 'Data_Percentage', 'Best_SSL', 'Mean']].rename(columns={'Mean': 'SSL_Mean'}),
        on=['Dataset', 'Data_Percentage']
    )
    comparison['SSL_Impact'] = comparison['SSL_Mean'] - comparison['Supervised_Mean']
    
    # Format for display
    comparison = comparison.sort_values(['Dataset', 'Data_Percentage'], ascending=[True, False])
    comparison['Data Percentage'] = comparison['Data_Percentage'].astype(str) + '%'
    comparison['SSL_Impact'] = comparison['SSL_Impact'].apply(lambda x: f"+{x:.1f}" if x > 0 else f"{x:.1f}")
    
    return comparison[['Dataset', 'Data Percentage', 'Best_Supervised', 'Best_SSL', 'SSL_Impact']]


# Run all functions
comparison_table = create_comparison_table(best_perf)
display(comparison_table)

# plot_top_backbones(best_perf)
# plot_all_interactive(best_perf)
# plot_best_worst_techniques(best_perf)

In [ ]:
combined_df

In [ ]:
plot_df = combined_df.groupby(['Dataset', 'Backbone', 'Technique', 'Data_Percentage'])['Mean'].agg(['mean','std']).reset_index()
# plot_df.columns = ['Dataset', 'Backbone', 'Technique', 'Data_Percentage', 'Mean', 'Std']
plot_df['Mean'] = plot_df['mean'] #* 100  # Convert to percentage                                                                                                                                                     
display(plot_df)

best_perf = get_best_performance(plot_df)
best_perf

In [ ]:
best_perf

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

def get_backbone_palette_plotly(df):
    """Create a Plotly-compatible color palette for backbones"""
    unique_backbones = df['Backbone'].unique()
    palette = sns.color_palette("husl", len(unique_backbones))
    return {backbone: f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' 
            for backbone, (r, g, b) in zip(unique_backbones, palette)}

def plot_technique_backbones_with_supervised(df):
    # Get Plotly-compatible color palette
    backbone_palette = get_backbone_palette_plotly(df)
    
    # Get unique datasets and techniques (excluding supervised)
    datasets = sorted(df['Dataset'].unique())
    techniques = sorted(df['Technique'].unique())
    # techniques = sorted([t for t in df['Technique'].unique() if t != 'Supervised'])
    
    # Get best supervised for each dataset
    best_supervised = df[df['Technique'] == 'Supervised']
    best_supervised = best_supervised.loc[best_supervised.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]

    best_ssl = df[df['Technique'] != 'Supervised']
    best_ssl = best_ssl.loc[best_ssl.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]
    
    # Create subplots - one row per dataset, one column per technique
    fig = make_subplots(
        rows=len(datasets), 
        cols=len(techniques),
        subplot_titles=[f"{d}<br>{t}" for d in datasets for t in techniques],
        shared_yaxes=True,
        vertical_spacing=0.05,
        horizontal_spacing=0.02
    )
    
    # Plot each dataset-technique combination
    for i, dataset in enumerate(datasets):
        for j, tech in enumerate(techniques):
            # Get all backbones for this technique-dataset combination
            tech_df = df[(df['Dataset'] == dataset) & (df['Technique'] == tech)]
            
            # Plot each backbone curve
            for backbone in tech_df['Backbone'].unique():
                backbone_df = tech_df[tech_df['Backbone'] == backbone].sort_values('Data_Percentage')
                # display(backbone_df)
                # a
                fig.add_trace(
                    go.Scatter(
                        x=backbone_df['Data_Percentage'],
                        y=backbone_df['Mean'],
                        mode='lines+markers',
                        name=backbone,
                        line=dict(color=backbone_palette[backbone]),
                        marker=dict(symbol='circle', size=8),
                        legendgroup=backbone,
                        showlegend=(i == 0 and j == 0),  # Only show legend for first subplot
                        hovertemplate=(
                            f"<b>Dataset:</b> {dataset}<br>"
                            f"<b>Technique:</b> {tech}<br>"
                            f"<b>Backbone:</b> {backbone}<br>"
                            f"<b>Samples per Class</b> %{{x}}<br>"
                            f"<b>Accuracy:</b> %{{y:.1f}}%<extra></extra>"
                            # f"<b>Dataset:</b> {backbone_df['Std']}<br>"
                        )
                    ),
                    row=i+1, col=j+1
                )
            
            # Add best supervised curve for this dataset
            sup_df = best_supervised[best_supervised['Dataset'] == dataset].sort_values('Data_Percentage')
            # display(sup_df)
            if not sup_df.empty:
                fig.add_trace(
                    go.Scatter(
                        x=sup_df['Data_Percentage'],
                        y=sup_df['Mean'],
                        mode='lines+markers',
                        name="Best Supervised",
                        line=dict(color='black', dash='dash'),
                        marker=dict(symbol='x', size=8),
                        showlegend=(i == 0 and j == 0),
                        hovertemplate=(
                            "<b>Best Supervised</b><br>"
                            f"<b>Samples per Class</b> %{{x}}<br>"
                            f"<b>Accuracy:</b> %{{y:.1f}}%<extra></extra>"
                            # f"<b>Dataset:</b> {sup_df['Std']}<br>"
                        )
                    ),
                    row=i+1, col=j+1
                )
            ssl_df = best_ssl[best_ssl['Dataset'] == dataset].sort_values('Data_Percentage')
            # display(ssl_df)
            # if not ssl_df.empty:
            #     fig.add_trace(
            #         go.Scatter(
            #             x=ssl_df['Data_Percentage'],
            #             y=ssl_df['Mean'],
            #             mode='lines+markers',
            #             name="Best of Dataset",
            #             line=dict(color='purple', dash='dash'),
            #             marker=dict(symbol='x', size=8),
            #             showlegend=(i == 0 and j == 0),
            #             hovertemplate=(
            #                 "<b>Best of Dataset</b><br>"
            #                 f"<b>Samples per Class</b> %{{x}}<br>"
            #                 f"<b>Accuracy:</b> %{{y:.1f}}%<extra></extra>"
            #                 # f"<b>Dataset:</b> {ssl_df['Std']}<br>"
            #                 # f"<b>SSL:</b> {ssl_df['Technique']}<br>"
            #                 # f"<b>Backbone:</b> {ssl_df['Backbone']}<br>"
            #             )
            #         ),
            #         row=i+1, col=j+1
            #     )
    
    # Update layout
    fig.update_layout(
        title=dict(
        text="Backbone Performance by LR and Dataset - Full Finetuning Supervised",
        font=dict(size=20),  # 🔼 Increase title font size here
        x=0.5,               # Center the title
        xanchor='center'
        ),
        height=500 * len(datasets),#250
        width=500 * len(techniques),#300
        hovermode="closest",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="right",
            x=1,
            font=dict(size=14)
        ),
        margin=dict(l=50, r=50, b=50, t=100, pad=4),
        plot_bgcolor='white'
    )
    
    # Update axes for all subplots
    for i in range(1, len(datasets)+1):
        for j in range(1, len(techniques)+1):
            fig.update_xaxes(
                type="log",
                tickvals=[1, 5, 10, 25, 50,100,200, 1000],
                ticktext=['1', '5', '10', '25', '50','100','200','max (100%)'],
                row=i, col=j
            )
            fig.update_yaxes(
                range=[0, 100],
                row=i, col=j
            )
            # Add grid lines
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey', row=i, col=j)
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey', row=i, col=j)
    
    # Add annotations
    # fig.add_annotation(
    #     x=0.5, y=1.1,
    #     xref="paper", yref="paper",
    #     text="Each subplot shows all backbones for a technique compared to the best supervised baseline",
    #     showarrow=False,
    #     font=dict(size=12)
    # )
    
    return fig

# Generate and show the plot
fig = plot_technique_backbones_with_supervised(best_perf)
fig.show()
fig.write_image("backbones_vs_supervised_lr3_seed.png")
fig.write_html("backbones_vs_supervised_lr3_seed.html")

In [ ]:
# 1. First, let's create a consistent color palette for backbones
def get_backbone_palette(df):
    unique_backbones = df['Backbone'].unique()
    palette = sns.color_palette("husl", len(unique_backbones))
    return dict(zip(unique_backbones, palette))

# 3. Enhanced best/worst techniques plot with additional markers
def plot_best_worst_techniques(df):
    # Get consistent color palette
    backbone_palette = get_backbone_palette(df)
    
    # Find best technique (excluding supervised) and worst technique
    non_supervised = df[df['Technique'] != 'Supervised']
    best_tech = non_supervised.loc[non_supervised.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]
    worst_tech = df.loc[df.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmin()]
    
    # Get best supervised for comparison
    supervised = df[df['Technique'] == 'Supervised']
    best_supervised = supervised.loc[supervised.groupby(['Dataset', 'Data_Percentage'])['Mean'].idxmax()]
    
    # Find overall best performance for each dataset
    overall_best = df.loc[df.groupby(['Dataset'])['Mean'].idxmax()]
    
    # Find when SSL reaches 90% and 95% of max performance
    ssl_90_95 = []
    for dataset in df['Dataset'].unique():
        max_acc = overall_best[overall_best['Dataset'] == dataset]['Mean'].values[0]
        ssl_data = non_supervised[non_supervised['Dataset'] == dataset]
        
        # Find first point where SSL reaches 90% and 95% of max
        for threshold in [0.90, 0.95]:
            threshold_acc = max_acc * threshold
            reached = ssl_data[ssl_data['Mean'] >= threshold_acc]
            if not reached.empty:
                first_reached = reached.loc[reached['Data_Percentage'].idxmin()]
                ssl_90_95.append({
                    'Dataset': dataset,
                    'Data_Percentage': first_reached['Data_Percentage'],
                    'Mean': first_reached['Mean'],
                    'Threshold': f"{int(threshold*100)}%",
                    'Backbone': first_reached['Backbone'],
                    'Technique': first_reached['Technique']
                })
    
    ssl_90_95_df = pd.DataFrame(ssl_90_95)
    
    # Combine with labels
    best_tech['Type'] = 'Best SSL'
    worst_tech['Type'] = 'Worst'
    best_supervised['Type'] = 'Best Supervised'
    combined = pd.concat([best_tech, worst_tech, best_supervised])
    
    # Plot styling
    sns.set_style("white")
    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 14,
        'axes.labelsize': 12,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11
    })
    
    # Create figure with subplots
    datasets = combined['Dataset'].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 6), sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    
    # Color palette
    palette = {'Best SSL': 'green', 'Worst': 'red', 'Best Supervised': 'blue'}
    
    # Plot each dataset
    for i, dataset in enumerate(datasets):
        ax = axes[i]
        subset = combined[combined['Dataset'] == dataset]
        dataset_best = overall_best[overall_best['Dataset'] == dataset]
        dataset_thresholds = ssl_90_95_df[ssl_90_95_df['Dataset'] == dataset]
        
        # Plot each type
        for typ, typ_df in subset.groupby('Type'):
            color = palette[typ]
            linestyle = '-' if typ != 'Best Supervised' else '--'
            
            # Plot line with markers
            ax.plot(typ_df['Data_Percentage'], typ_df['Mean'], 
                   color=color, marker='o', linestyle=linestyle,
                   label=typ, linewidth=2)
            
            # Add annotations for backbone and technique
            for _, row in typ_df.iterrows():
                if typ == 'Best Supervised':
                    label = f"{row['Backbone']}"
                else:
                    label = f"{row['Backbone']}\n{row['Technique']}"
                
                y_offset = 15 if typ == 'Best SSL' else (-25 if typ == 'Worst' else 0)
                ax.annotate(label,
                           (row['Data_Percentage'], row['Mean']),
                           textcoords="offset points",
                           xytext=(0, y_offset),
                           ha='center', fontsize=9,
                           bbox=dict(boxstyle='round,pad=0.3', 
                                    fc='white', alpha=0.8))
        
        # Mark overall best performance
        if not dataset_best.empty:
            ax.scatter(dataset_best['Data_Percentage'], dataset_best['Mean'],
                      marker='*', s=200, color='gold', zorder=5,
                      label='Best Overall')
        
        # Mark 90% and 95% thresholds
        for _, row in dataset_thresholds.iterrows():
            marker = '^' if row['Threshold'] == '90%' else 'v'
            color = 'purple' if row['Threshold'] == '90%' else 'blue'
            size = 100 if row['Threshold'] == '90%' else 80
            label = f"Reached {row['Threshold']} of max"
            
            ax.scatter(row['Data_Percentage'], row['Mean'],
                      marker=marker, s=size, color=color, zorder=5,
                      label=label)

        # Formatting
        ax.set_xscale('log')
        ax.set_xticks([1, 5, 10, 25, 50,100,200, 1000])
        ax.set_xticklabels(['1', '5', '10', '25', '50','100','200','max (100%)'])
        ax.set_xlabel('Samples per Class', fontweight='bold')
        ax.set_ylabel('Balanced Accuracy (%)' if i == 0 else '', fontweight='bold')
        ax.set_title(dataset, fontweight='bold')
        ax.set_ylim(0, 100)
        ax.grid(True, which="both", ls="--", alpha=0.2)
    
    # Single legend at the top
    handles = [
        Line2D([0], [0], color=palette['Best SSL'], marker='o', linestyle='-', label='Best Supervised'),
        # Line2D([0], [0], color=palette['Best Supervised'], marker='o', linestyle='--', label='Best Supervised'),
        Line2D([0], [0], color=palette['Worst'], marker='o', linestyle='-', label='Worst'),
        Line2D([0], [0], marker='*', color='gold', linestyle='None', markersize=10, label='Best Overall'),
        Line2D([0], [0], marker='^', color='purple', linestyle='None', markersize=10, label='Reached 90% of max'),
        Line2D([0], [0], marker='v', color='blue', linestyle='None', markersize=10, label='Reached 95% of max')
    ]
    
    fig.legend(handles=handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.1))
    
    plt.tight_layout()
    plt.savefig('best_worst_techniques_enhanced_lr3_seed.png', format='png', bbox_inches='tight', dpi=300)
    plt.show()
plot_best_worst_techniques(best_perf)

### how can we view this worst cases, best supervised and best ssl?

In [ ]:
# use explorando results.ipynb
breakpoint

### P1. Qual o melhor backbone geral para HAR?

Qual o melhor backbone para HAR em SSL?
Qual o melhor backbone para HAR em SL?


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ssl_df = df[df["tsk_pretext"] != "Supervised"]
supervised_df = df[df["tsk_pretext"] == "Supervised"]
# Define a custom palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-1D": "tab:green",
    "CNN PF": "tab:red",
    'TS Encoder': 'tab:purple'
}
# Define the order of backbones
backbone_order = ["RNN", "IMU Transformer", "ResNet-1D", "CNN PF",'TS Encoder']
# Get colors from Viridis
viridis_colors = sns.color_palette("deep", n_colors=len(backbone_order))

# Create a dictionary mapping backbone names to Viridis colors
viridis_palette = {backbone: color for backbone, color in zip(backbone_order, viridis_colors)}

# Prepare the data as percentage
ssl_df_percent = ssl_df.copy()
supervised_df_percent = supervised_df.copy()
ssl_df_percent["metric"] *= 100
supervised_df_percent["metric"] *= 100

# Determine y-axis limits
ymin = min(ssl_df_percent["metric"].min(), supervised_df_percent["metric"].min())
ymax = max(ssl_df_percent["metric"].max(), supervised_df_percent["metric"].max())
ylims = (max(0, ymin - 5), min(100, ymax + 5))

# Clean aesthetic with larger font
sns.set(style="whitegrid", font_scale=1.5)
custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "black",  # Change grid line color
    "grid.linestyle": "--",  # Change grid line style
    "grid.linewidth": 3  # Change grid line width
}
sns.set_context("paper", rc=custom_params)

label_fontsize = 14
tick_fontsize = 12

# SSL Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x='backbone', y='metric', data=ssl_df_percent, order=backbone_order, palette=viridis_palette)
plt.xlabel("Backbone", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(ylims)
plt.tight_layout()
plt.savefig('ssl_rq1.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Supervised Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x='backbone', y='metric', data=supervised_df_percent, order=backbone_order, palette=viridis_palette)
plt.xlabel("Backbone", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(ylims)
plt.tight_layout()
plt.savefig('supervised_rq1.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()



## Note:
 use show_df_tests=True, to see intermediate results

In [ ]:
df

In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_ssl_rq1"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

summary_df_ssl_rq12




In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "Supervised"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_sl_rq1"
    
)




summary_df_supervised_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_supervised_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + Supervised'

summary_df_supervised_rq12




In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Filter relevant columns
df_filtered = df[["backbone", "tsk_pretext", "metric"]]

# Aggregate mean metric values per backbone and task
df_grouped = df_filtered.groupby(["backbone", "tsk_pretext"], as_index=False).mean()

# Rename columns
df_grouped.rename(columns={"metric": "Score"}, inplace=True)

# Sort the dataframe by Score within each tsk_pretext
df_grouped = df_grouped.sort_values(by=["tsk_pretext", "Score"], ascending=[True, False])
# df_grouped["Accuracy"] *= 100

# Plot
# Create the plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x="tsk_pretext", y="Score", hue="backbone", data=df_grouped, 
                 palette=viridis_palette, errwidth=1.5, capsize=0.05)


plt.xlabel("Task Pretext", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=45, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(0, 1)

# Legend styling
legend = plt.legend(title="Backbone", bbox_to_anchor=(1, 1), 
                   fontsize=12)
plt.setp(legend.get_title(), fontsize=12, fontweight='bold')

# Adjust layout and save
plt.tight_layout()
plt.savefig('ssl_sl_rq2.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:


finetune_df = df[df["ft_strategy"] == "Full Finetune"]
freeze_df = df[df["ft_strategy"] == "Freeze"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
custom_palette = {
    "RNN": "blue",
    "IMU Transformer": "red",
    "ResNet-1D": "green",
    "CNN PF": "orange",
    "TS Encoder": "purple"
}



# Convert frac_dtarget to categorical and update to percentage strings
# try:
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(float) * 100
#     # finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(str) + '%'
# except ValueError as e:
#     print(f"Error converting frac_dtarget: {e}")

# Group by backbone and frac_dtarget, and calculate mean accuracy and standard deviation
mean_df = finetune_df.groupby(['backbone','frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
display(mean_df)
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].astype(str)
# Set enhanced style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",  # Bold axis labels
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.titlesize": 14,        # Larger title (though we'll remove it)
})

# Create figure with larger dimensions
plt.figure(figsize=(11, 7), layout='constrained')

# Create pointplot with enhanced visibility
ax = sns.pointplot(
    data=mean_df, 
    x='frac_dtarget', 
    y='mean', 
    hue='backbone',
    # palette=viridis_palette,
    hue_order=["RNN", "IMU Transformer", "ResNet-1D", "CNN PF", "TS Encoder"],
    order=["1.0", "5.0", "10.0", "50.0", "100.0","200.0","1000.0"],
    errwidth=2.0,      # Thicker error bars
    capsize=0.15,      # Slightly larger caps
    # markersize=10      # Larger points
)

# Enhanced axis labels
plt.xlabel("Data Percentage", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)
plt.ylabel("Balanced Accuracy", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)

# Bold axis ticks with larger font
plt.xticks(rotation=45, 
          fontsize=13,  # Increased from 11
          fontweight='bold')
plt.yticks(fontsize=13,  # Increased from 11
          fontweight='bold')

# Y-axis formatting
plt.ylim(0.3, 1.0)
ax.set_yticklabels(['{:.0f}%'.format(y*100) for y in ax.get_yticks()])

# Enhanced grid
plt.grid(True, alpha=0.7)

# Upgraded legend
handles, labels = ax.get_legend_handles_labels()
legend = plt.legend(
    handles, 
    labels, 
    title="Backbone",
    title_fontsize=18,  # Increased from 12
    fontsize=16,        # Increased from 11
    bbox_to_anchor=(1, 1),
    loc='upper left',
    frameon=True,
    framealpha=1,
    edgecolor='black'
)

# Make legend title bold
legend.get_title().set_fontweight('bold')

# Save high-quality output
plt.savefig('finetune_rq4.png', 
           dpi=350,      # Higher than standard 300
           bbox_inches='tight', 
           transparent=True)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


plot_df = df.copy()
plot_df["metric"] *= 100

# Group by backbone and pretext task
# df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False).mean()
# df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False)["metric"].mean()

# df_grouped.rename(columns={"metric": "Score"}, inplace=True)

# Calculate mean and std
df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False)["metric"].agg(["mean", "std"]).reset_index()
df_grouped.rename(columns={"mean": "Score", "std": "Error"}, inplace=True)

df_grouped = df_grouped.sort_values(by=["tsk_pretext", "Score"], ascending=[True, False])

# Custom backbone order and Viridis palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:red",
    "ResNet-1D": "tab:green",
    "CNN PF": "tab:orange",
    # 'TS Encoder': 'tab:purple'
}
# Define the order of backbones
backbone_order = ["RNN", "IMU Transformer", "ResNet-1D", "CNN PF"]#,'TS Encoder']
# Get colors from Viridis
viridis_colors = sns.color_palette("deep", n_colors=len(backbone_order))
viridis_palette = {backbone: color for backbone, color in zip(backbone_order, viridis_colors)}

# Aesthetic settings
sns.set(style="whitegrid", font_scale=1.5)
custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "black",
    "grid.linestyle": "--",
    "grid.linewidth": 1.5
}
sns.set_context("paper", rc=custom_params)

label_fontsize = 18
tick_fontsize = 16

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x="tsk_pretext",
    y="Score",
    hue="backbone",
    data=df_grouped,
    hue_order=backbone_order,
    palette=viridis_palette,
    ci=None,           # Disable built-in CI
    errwidth=1.5,
    capsize=0.05
)

# Manually add error bars
for i, bar in enumerate(ax.patches):
    # Match error bar to correct data point
    group_index = i // len(backbone_order)
    within_group_index = i % len(backbone_order)
    subset = df_grouped[df_grouped["tsk_pretext"] == df_grouped["tsk_pretext"].unique()[group_index]]
    error = subset.iloc[within_group_index]["Error"]
    height = bar.get_height()
    ax.errorbar(
        bar.get_x() + bar.get_width() / 2,
        height,
        yerr=error,
        ecolor='black',
        capsize=5,
        fmt='none'
    )


plt.xlabel("Pretext Task", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(0, 100)

# Legend styling
legend = plt.legend(
    title="Backbone",
    bbox_to_anchor=(0.5, 1.15),  # center and slightly above
    loc="lower center",
    fontsize=14,
    ncol=len(backbone_order)    # one column per legend entry
)
plt.setp(legend.get_title(), fontsize=14, fontweight='bold')


plt.tight_layout()
plt.savefig("ssl_sl_rq2.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
# all backbones for TNC
dot,df_tnc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TNC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tnc_rq2"
    
)

df_tnc

summary_df_tnc = summarize_backbone_performance(df_tnc)
display(summary_df_tnc)
# all backbones for TFC
dot,df_tfc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TFC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tfc_rq2"
    
)

df_tfc

summary_df_tfc = summarize_backbone_performance(df_tfc)
display(summary_df_tfc)
# all backbones for DIET
dot,df_diet = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["Diet"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_diet_rq2"
    
)

df_diet

summary_df_diet = summarize_backbone_performance(df_diet)
display(summary_df_diet)
# all backbones for LFR
dot,df_lfr = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["LFR"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_lfr_rq2"
    
)

df_lfr

summary_df_lfr = summarize_backbone_performance(df_lfr)
display(summary_df_lfr)
# Combine all DataFrames
combined_df_rq2 = pd.concat([
    summary_df_supervised_rq12,
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
], ignore_index=False)

# Compute total wins/losses per backbone across all techniques
summary_df_rq2 = combined_df_rq2.groupby("Backbone").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Mean": "mean",    # Average mean across all techniques
    "Std": "mean"      # Average std across all techniques
}).reset_index()

# Calculate Net Score
summary_df_rq2["Net Score"] = summary_df_rq2["Wins"] - summary_df_rq2["Losses"]
summary_df_rq2["Mean"] = summary_df_rq2["Mean"] * 100
summary_df_rq2["Std"] = summary_df_rq2["Std"] * 100
# Format Mean ± Std (e.g., "75.9 ± 1.3")
summary_df_rq2["Performance"] = (
    summary_df_rq2["Mean"].round(1).astype(str) + 
    "% ± " + 
    summary_df_rq2["Std"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
summary_df_rq2.sort_values(
    ["Net Score", "Mean"], 
    ascending=[False, False], 
    inplace=True
)

# Reorder columns for clarity
summary_df_rq2 = summary_df_rq2[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(summary_df_rq2)

# Split the backbone name and create new column
summary_df_rq2["Backbone Only"] = summary_df_rq2["Backbone"].str.split(" \+ ").str[0]

# Extract numeric values from Performance column for calculations
summary_df_rq2[['Mean_pct', 'Std_pct']] = summary_df_rq2['Performance'].str.extract(r'(\d+\.\d+)% ± (\d+\.\d+)%').astype(float)

# Group by Backbone and aggregate all metrics
backbone_totals = summary_df_rq2.groupby("Backbone Only").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Net Score": "sum",
    "Mean_pct": "mean",  # Average mean across all variants
    "Std_pct": "mean"    # Average std across all variants
}).reset_index()

# Format Performance column with percentage
backbone_totals["Performance"] = (
    backbone_totals["Mean_pct"].round(1).astype(str) + 
    "% ± " + 
    backbone_totals["Std_pct"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
backbone_totals = backbone_totals.sort_values(
    ["Net Score", "Mean_pct"], 
    ascending=[False, False]
)

# Rename and clean up columns
backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
backbone_totals.replace({"Backbone": {
    'TS2Vec': 'TS2Vec (Partial)',
}}, inplace=True)

# Select final columns
backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(backbone_totals)



In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals

# Assuming you have these individual technique DataFrames:
technique_dfs = [
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
]

technique_names = [
    "TNC",
    "TFC",
    "Diet",
    "LFR"
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# all combined
technique_dfs = [summary_df_ssl_rq12,
summary_df_supervised_rq12,
summary_df_tnc,
summary_df_tfc,
summary_df_diet,
summary_df_lfr]

technique_names = [
    "SSL",
    "SSL with TS2Vec",
    "Supervised"
    "TNC",
    "TFC",
    "Diet",
    "LFR"
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

backbones cnn better but a lot of variance we need to take a close look on whats happening

### P3. Qual o melhor backbone geral para HAR de acordo com a estratégia de refino?

Qual o melhor backbone para HAR usando a estratégia de refino Freeze?
Qual o melhor backbone para HAR usando a estratégia de refino Finetune?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "#DDDDDD",
    "grid.linewidth": 1.2
})

# Aesthetic scaling
sns.set_context("paper", rc={"axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14})

# Create figure
plt.figure(figsize=(10, 6))

# Boxplot
ax = sns.boxplot(
    x='ft_strategy', 
    y='metric', 
    hue='backbone', 
    data=plot_df, 
    hue_order=backbone_order, 
    palette=viridis_palette,
    linewidth=1.6,
    fliersize=3,
    width=0.7
)

# Labels & title
# plt.title("Impact of Fine-Tuning Strategy on Model Accuracy", 
#           fontsize=16, pad=14, fontweight='bold')
plt.xlabel("Fine-Tuning Strategy", fontsize=18, labelpad=12, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=18, labelpad=12, fontweight='bold')

# Format ticks
rotation_angle = 30 if len(df['ft_strategy'].unique()) > 3 else 0
plt.xticks(rotation=rotation_angle, fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.ylim(0, 100)

# Legend
legend = plt.legend(
    title="Backbone",
    title_fontsize=13,
    fontsize=12,
    bbox_to_anchor=(0.5, 1.15),
    loc='lower center',
    ncol=len(backbone_order),
    frameon=True
)
plt.setp(legend.get_title(), fontweight='bold')

# Layout
plt.tight_layout()
plt.savefig("ssl_sl_rq3.png", dpi=300, bbox_inches='tight', transparent=False)
plt.show()
plt.close()


In [ ]:
# FULL FINETUNE with TS2VEC for available techniques
dot,df_finetune_rq3_ts2vec = show_precedence_graph(
    df,
    variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
        # "select_tsk_pretext": ["Supervised", "TNC","Diet","TFC","LFR"],
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_finetune_rq3_with_ts2vec"
    
)

summary_df_finetune_rq3_ts2vec= summarize_backbone_performance(df_finetune_rq3_ts2vec)
display(summary_df_finetune_rq3_ts2vec)

# FREEZE with TS2VEC for available techniques

dot,df_freeze_rq3_with_ts2vec = show_precedence_graph(
    df,
    variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
        "select_tsk_pretext": ["Supervised", "TNC","Diet","TFC","LFR"],
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_freeze_rq3_with_ts2vec"
    
)


summary_df_freeze_rq3_with_ts2vec= summarize_backbone_performance(df_freeze_rq3_with_ts2vec)
display(summary_df_freeze_rq3_with_ts2vec)





### P4. Qual o melhor backbone geral para HAR de acordo com a porcentagem de dados em caso de Finetune?

Qual o melhor backbone para refino com  1% dos dados?
Qual o melhor backbone para refino com  5% dos dados?
Qual o melhor backbone para refino com  10% dos dados?
Qual o melhor backbone para refino com  50% dos dados?
Qual o melhor backbone para refino com  100% dos dados?



In [ ]:


finetune_df = df[df["ft_strategy"] == "Full Finetune"]
freeze_df = df[df["ft_strategy"] == "Freeze"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert frac_dtarget to percentage string labels
# try:
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(float) * 100
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(str) + '%'
# except Exception as e:
#     print("Error converting frac_dtarget to percentage:", e)

# Aggregate mean and std
mean_df = finetune_df.groupby(['backbone', 'frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()

# Seaborn style
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "#e0e0e0",
    "grid.linewidth": 1.2
})
sns.set_context("paper", rc={"axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14})

# Create plot
plt.figure(figsize=(14, 7), layout='constrained')  # Increased width for more space

# Create barplot
ax = sns.barplot(
    data=mean_df,
    x='frac_dtarget',
    y='mean',
    hue='backbone',
    palette=viridis_palette,
    hue_order=backbone_order,
    order=["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"],
)

# Add error bars
for container, backbone in zip(ax.containers, backbone_order):
    # Get the std values for the corresponding backbone
    subset = mean_df[mean_df['backbone'] == backbone]
    std_values = subset['std'].values
    
    for bar, std in zip(container, std_values):
        height = bar.get_height()
        x = bar.get_x() + bar.get_width() / 2
        plt.errorbar(
            x=x,
            y=height,
            yerr=std,
            fmt='none',
            c='black',
            capsize=5,
            elinewidth=2,
            alpha=0.7,
            zorder=10
        )

# Axis labels
plt.xlabel("Fraction of Target Data (%)", fontsize=16, fontweight='bold', labelpad=12)
plt.ylabel("Balanced Accuracy (%)", fontsize=16, fontweight='bold', labelpad=12)

# Ticks
plt.xticks(rotation=30, fontsize=13, fontweight='bold')
plt.yticks(fontsize=13, fontweight='bold')

# Y ticks to percentage
ax.set_yticklabels([f"{y * 100:.0f}%" for y in ax.get_yticks()])

# Limits (set from 0 to 100%)
plt.ylim(0, 1.0)

# Grid
plt.grid(True, alpha=0.8)

# Legend
legend = plt.legend(
    title="Backbone",
    title_fontsize=16,
    fontsize=14,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.17),
    ncol=5,
    frameon=True
)
plt.setp(legend.get_title(), fontweight='bold')

# Save
plt.savefig('finetune_rq4_barplot.png', dpi=350, bbox_inches='tight', transparent=True)
plt.show()
plt.close()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
custom_palette = {
    "RNN": "blue",
    "IMU Transformer": "red",
    "ResNet-1D": "green",
    "CNN PF": "orange",
    "TS Encoder": "purple"
}



# Convert frac_dtarget to categorical and update to percentage strings
# try:
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(float) * 100
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(str) + '%'
# except ValueError as e:
#     print(f"Error converting frac_dtarget: {e}")

# Group by backbone and frac_dtarget, and calculate mean accuracy and standard deviation
mean_df = finetune_df.groupby(['backbone','frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
display(mean_df)
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].astype(str)
# Set enhanced style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",  # Bold axis labels
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.titlesize": 14,        # Larger title (though we'll remove it)
})

# Create figure with larger dimensions
plt.figure(figsize=(11, 7), layout='constrained')

# Create pointplot with enhanced visibility
ax = sns.pointplot(
    data=mean_df, 
    x='frac_dtarget', 
    y='mean', 
    hue='backbone',
    # palette=viridis_palette,
    hue_order=["RNN", "IMU Transformer", "ResNet-1D", "CNN PF", "TS Encoder"],
    order=["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"],
    errwidth=2.0,      # Thicker error bars
    capsize=0.15,      # Slightly larger caps
    # markersize=10      # Larger points
)

# Enhanced axis labels
plt.xlabel("Number of Samples", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)
plt.ylabel("Balanced Accuracy", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)

# Bold axis ticks with larger font
plt.xticks(rotation=45, 
          fontsize=13,  # Increased from 11
          fontweight='bold')
plt.yticks(fontsize=13,  # Increased from 11
          fontweight='bold')

# Y-axis formatting
plt.ylim(0.3, 1.0)
ax.set_yticklabels(['{:.0f}%'.format(y*100) for y in ax.get_yticks()])

# Enhanced grid
plt.grid(True, alpha=0.7)

# Upgraded legend
handles, labels = ax.get_legend_handles_labels()
legend = plt.legend(
    handles, 
    labels, 
    title="Backbone",
    title_fontsize=18,  # Increased from 12
    fontsize=16,        # Increased from 11
    bbox_to_anchor=(1, 1),
    loc='upper left',
    frameon=True,
    framealpha=1,
    edgecolor='black'
)

# Make legend title bold
legend.get_title().set_fontweight('bold')

# Save high-quality output
plt.savefig('finetune_rq42.png', 
           dpi=350,      # Higher than standard 300
           bbox_inches='tight', 
           transparent=True)

plt.show()

### ok, finetuning is better than freeze, than lets do statistical tests for each dataset percentage

In [ ]:
# Define the target fractions and their display names
target_fractions = ["1.0", "5.0", "10.0", "25.0", "50.0", "100.0", "200.0", "1000.0"]
display_names = [
    "Finetune 1", 
    "Finetune 5",
    "Finetune 10",
    "Finetune 25",
    "Finetune 50",
    "Finetune 100",
    "Finetune 200",
    "Finetune 1000"
]

# Initialize lists to store results
technique_dfs = []
technique_names = []
raw_dfs = []  # Optional: if you need the raw DataFrames too

for frac, name in zip(target_fractions, display_names):
    print(f"\n{'='*50}")
    print(f"Analyzing: {name}")
    print(f"{'='*50}")
    
    # Create the percentage string
    frac_percent = f"{frac}"
    
    # Generate the precedence graph and results
    dot, df_finetune = show_precedence_graph(
        df,
        variants_variables=["backbone", "frac_dtarget"],
        filters={
            "select_frac_dtarget": [frac_percent],
            "select_ft_strategy": ["Full Finetune"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    
    # Summarize backbone performance
    summary_df = summarize_backbone_performance(df_finetune)
    
    # Store results in lists
    technique_dfs.append(summary_df)
    technique_names.append(name)
    raw_dfs.append(df_finetune)  # Optional
    
    # Display results for current fraction
    display(summary_df)
    
    # Optionally save the dot graph
    # dot.render(f'precedence_graph_{name.replace(" ", "_")}', format='png', cleanup=True)

# Now you have:
# technique_dfs - list of summary DataFrames in order
# technique_names - corresponding display names
# raw_dfs - optional list of raw DataFrames (uncomment if needed)

# Example usage:
for df, name in zip(technique_dfs, technique_names):
    print(f"\n{name} Results:")
    display(df.head())

In [ ]:
# Assuming you have these individual technique DataFrames:
# technique_dfs = [
#     summary_df_finetune_rq4_100,
# summary_df_finetune_rq4_50,
# summary_df_finetune_rq4_10,
# summary_df_finetune_rq4_5,
# summary_df_finetune_rq4_1
# ]

# technique_names = [
#     "Finetune 100%",
#     "Finetune 50%",
#     "Finetune 10%",
#     "Finetune 5%",
#     "Finetune 1%"
# ]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

CNNs e Resnet bem mais significante em 5 e 10% dos dados

### P5. Qual o melhor backbone para cada técnica de acordo com a porcentagem de dados em caso de finetune?

Qual o melhor backbone para refino com  1% dos dados usando cada técnica?
Qual o melhor backbone para refino com  5% dos dados usando cada técnica?
Qual o melhor backbone para refino com  10% dos dados usando cada técnica?
Qual o melhor backbone para refino com  50% dos dados usando cada técnica?
Qual o melhor backbone para refino com  100% dos dados usando cada técnica?


In [ ]:
df = base_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ======================
# SETUP & CONFIGURATION
# ======================
# Define custom color palette with consistent ordering
# backbone_order = ["CNN", "ResNet", "Transformer", "RNN", "TS2Vec"]
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-1D": "tab:green",
    "CNN PF": "tab:red",
    'TS Encoder': 'tab:purple'
}

# Configure global plot style
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
})

# ======================
# DATA PREPARATION
# ======================
# Filter and process data
df = df[df["ft_strategy"] == "Full Finetune"]
frac_order = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]

# Split data
ssl = df[df["tsk_pretext"] != "Supervised"].copy()
supervised = df[df["tsk_pretext"] == "Supervised"].copy()

# Convert to categorical for proper ordering
ssl["frac_dtarget"] = pd.Categorical(ssl["frac_dtarget"], categories=frac_order, ordered=True)
supervised["frac_dtarget"] = pd.Categorical(supervised["frac_dtarget"], categories=frac_order, ordered=True)

# ======================
# PLOTTING FUNCTION
# ======================
def create_facetgrid(data, title_suffix, filename, legend=False):
    """Create standardized facet grid plot"""
    g = sns.FacetGrid(
        data, 
        col="d_target", 
        col_wrap=3, 
        height=4,
        aspect=1.2,
        despine=True,
        sharey=True
    )
    
    # Create line plots
    g.map_dataframe(
        sns.lineplot,
        x="frac_dtarget",
        y="metric",
        hue="backbone",
        hue_order=backbone_order,
        palette=custom_palette,
        marker="o",
        markersize=8,
        linewidth=2.5,
        errorbar=('ci', 95),
        err_style="bars",
        err_kws={"capsize": 4, "capthick": 1.5, "elinewidth": 1.5}
    )
    
    # Set axis labels
    g.set_axis_labels(
        "Fraction of Target Data (%)",  # Added % to label
        "Accuracy (%)",
        fontsize=12,
        fontweight='bold'
    )
    
    # Configure ticks
    for ax in g.axes.flat:
        # Ensure x-axis shows all categories
        ax.set_xticks(range(len(frac_order)))
        ax.set_xticklabels(frac_order, rotation=45, ha='right')
        
        # Convert y-ticks to percentages
        yticks = ax.get_yticks()
        ax.set_yticklabels([f"{y*100:.0f}%" for y in yticks])
    
    # Add legend
    if legend:
        g.add_legend(
            title="Backbone",
            title_fontsize=12,
            fontsize=11,
            bbox_to_anchor=(1.05, 0.5),
            loc='center left',
            frameon=True,
            framealpha=1,
            edgecolor='black'
        )
    
    # Adjust layout
    g.fig.subplots_adjust(top=0.88, right=0.85 if legend else 0.95, 
                         hspace=0.3, wspace=0.2)
    
    # Add title
    # g.fig.suptitle(
    #     f"Performance Across Data Regimes: {title_suffix}",
    #     y=0.95,
    #     fontsize=14,
    #     fontweight='bold'
    # )
    
    # Save with quality settings
    plt.savefig(
        filename,
        dpi=300,
        bbox_inches='tight',
        transparent=True,
        facecolor='white'
    )
    plt.close()
# ======================
# GENERATE PLOTS
# ======================
create_facetgrid(
    ssl, 
    "SSL Pretraining (Full Finetune)", 
    "rq4_ssl_full_finetune.png",
    # legend=True
)

create_facetgrid(
    supervised, 
    "Supervised Baseline (Full Finetune)", 
    "rq4_supervised_full_finetune.png",
    # legend=True
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Custom color palette
# custom_palette = {
#     "RNN": "tab:blue",
#     "Transformer": "tab:orange",
#     "ResNet": "tab:green",
#     "CNN": "tab:red",
#     "TS2Vec": "tab:purple"
# }

# Filter only Full Finetune
df_full = df[df["ft_strategy"] == "Full Finetune"].copy()

# Rename datasets
df_full["d_target"] = df_full["d_target"].replace({
    "MS": "MotionSense",
    "KH": "Kuhar"
})

# Set dataset order
dataset_order = ["UCI", "MotionSense", "RW-Thigh", "RW-Waist", "Kuhar", "WISDM"]
df_full["d_target"] = pd.Categorical(df_full["d_target"], categories=dataset_order, ordered=True)

# Split SSL and Supervised
ssl = df_full[df_full["tsk_pretext"] != "Supervised"].copy()
supervised = df_full[df_full["tsk_pretext"] == "Supervised"].copy()

# Fraction order
frac_order = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]

# Set frac_dtarget as ordered categorical
ssl["frac_dtarget"] = pd.Categorical(ssl["frac_dtarget"], categories=frac_order, ordered=True)
supervised["frac_dtarget"] = pd.Categorical(supervised["frac_dtarget"], categories=frac_order, ordered=True)

# Define a function to plot
def plot_finetune(data, save_path):
    g = sns.FacetGrid(
        data,
        col="d_target",
        col_order=dataset_order,  # <--- important for correct order
        col_wrap=3,
        height=4,
        sharey=True,
        sharex=True,
        despine=True
    )

    g.map_dataframe(
        sns.lineplot,
        x="frac_dtarget",
        y="metric",
        hue="backbone",
        marker="o",
        palette=custom_palette
    )

    # Remove per-axes labels
    g.set_titles("{col_name}")
    g.set(xlabel=None, ylabel=None)
    g.set(ylim=(0, 1.0))

    # Format y-axis as percentage
    for ax in g.axes.flatten():
        yticks = ax.get_yticks()
        ax.set_yticklabels([f"{y * 100:.0f}%" for y in yticks])

    # Handle legend manually
    handles, labels = g.axes[0].get_legend_handles_labels()
    if g._legend:
        g._legend.remove()

    g.fig.legend(
        handles,
        labels,
        title="Backbone",
        title_fontsize=16,
        fontsize=14,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),  # closer to plot
        ncol=len(custom_palette),
        frameon=True
    )

    # Add unified x and y labels
    g.fig.text(0.5, 0.005, "Fraction of Target Data (%)", ha="center", va="center", fontsize=16, fontweight="bold")
    g.fig.text(0.005, 0.5, "Balanced Accuracy (%)", ha="center", va="center", rotation="vertical", fontsize=16, fontweight="bold")

    # Adjust layout
    g.fig.subplots_adjust(top=0.85, bottom=0.1)

    plt.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()

# Now call the function
plot_finetune(
    ssl,
    "rq4_ssl_full_finetune.png"
)

plot_finetune(
    supervised,
    "rq4_supervised_full_finetune.png"
)


In [ ]:
# Define custom color palette
# custom_palette = {
#     "RNN": "tab:blue",
#     "Transformer": "tab:orange",
#     "ResNet": "tab:green",
#     "CNN": "tab:red",
#     "TS2Vec": "tab:purple"
# }

df = df[df["ft_strategy"] == "Full Finetune"]  # Focus on full finetuning

# Split SSL and supervised data
ssl = df[df["tsk_pretext"] != "Supervised"]
supervised = df[df["tsk_pretext"] == "Supervised"]

frac_order = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]

# Convert `frac_dtarget` to a categorical variable with the desired order
ssl["frac_dtarget"] = pd.Categorical(ssl["frac_dtarget"], categories=frac_order, ordered=True)

# Create the FacetGrid
g = sns.FacetGrid(ssl, col="d_target", col_wrap=3, height=4)

# Map the lineplot with the ordered `frac_dtarget` and custom colors
g.map(sns.lineplot, "frac_dtarget", "metric", "backbone", marker="o", palette=custom_palette)

# Add legend
g.add_legend()

# Set the main title
g.fig.suptitle("Accuracy vs. Fraction of Target Data for Full Finetune of Different Backbones (SSL)", fontsize=16)

# Adjust layout to prevent overlap
g.fig.subplots_adjust(top=0.85)
plt.savefig("rq4_ssl_full_finetune.png", dpi=300, bbox_inches='tight')
plt.show()

# Convert `frac_dtarget` to a categorical variable with the desired order
supervised["frac_dtarget"] = pd.Categorical(supervised["frac_dtarget"], categories=frac_order, ordered=True)

# Create the FacetGrid
g = sns.FacetGrid(supervised, col="d_target", col_wrap=3, height=4)

# Map the lineplot with the ordered `frac_dtarget` and custom colors
g.map(sns.lineplot, "frac_dtarget", "metric", "backbone", marker="o", palette=custom_palette)

# Add legend
g.add_legend()

# Set the main title
g.fig.suptitle("Accuracy vs. Fraction of Target Data for Full Finetune of Different Backbones (Supervised)", fontsize=16)

# Adjust layout to prevent overlap
g.fig.subplots_adjust(top=0.85)
plt.savefig("rq4_supervised_full_finetune.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-1D": "tab:green",
    "CNN PF": "tab:red",
    'TS Encoder': 'tab:purple'
}
# List of unique SSL tasks
tasks = finetune_df["tsk_pretext"].unique()
frac_order = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]
finetune_df["frac_dtarget"] = pd.Categorical(
    finetune_df["frac_dtarget"],
    categories=frac_order,
    ordered=True
)
# Plot each task separately
for task in tasks:
    plt.figure(figsize=(10, 6))
    task_data = finetune_df[finetune_df["tsk_pretext"] == task]
    
    # Check unique values in frac_dtarget for the current task
    unique_frac_dtarget = task_data["frac_dtarget"].unique()
    print(f"Task: {task}, Unique frac_dtarget: {unique_frac_dtarget}")
    
    sns.pointplot(
        data=task_data,
        x="frac_dtarget",
        y="metric",
        hue="backbone",
        hue_order=["RNN", "IMU Transformer", "ResNet-1D", "CNN PF","TS Encoder"],
        # order=["1.0%", "5.0%", "10.0%", "50.0%", "100.0%"],
        # markers=["o", "s", "D", "^"],  # Provide four markers
        # linestyles=["-", "--", "-.", ":"],  # Provide four linestyles
        scale=0.8,
        palette=custom_palette
    )
    plt.title(f"Task Pretext: {task}")
    plt.xlabel("Fraction of Target Data (%)")
    plt.ylabel("Accuracy")
    plt.ylim(0.1, 1)
    plt.legend(title="Backbone")
    plt.tight_layout()
    plt.show()

### Now lets see full finetune for each technique and percentage

In [ ]:
tnc_dfs = {}
for pct in ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]:
    # pct_clean = pct.replace(".", "").replace("%", "")
    dot, df_result = show_precedence_graph(
        df,
        variants_variables=["backbone","frac_dtarget","tsk_pretext"],
        filters={
            "select_frac_dtarget": [pct],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TNC"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    tnc_dfs[f"df_finetune_tnc_rq4_{pct}"] = df_result
    summary = summarize_backbone_performance(df_result)
    display(summary)


In [ ]:
diet_dfs = {}
for pct in ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]:
    # pct_clean = pct.replace(".", "").replace("%", "")
    dot, df_result = show_precedence_graph(
        df,
        variants_variables=["backbone","frac_dtarget","tsk_pretext"],
        filters={
            "select_frac_dtarget": [pct],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Diet"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    diet_dfs[f"df_finetune_diet_rq4_{pct}"] = df_result
    summary = summarize_backbone_performance(df_result)
    display(summary)

tfc_dfs = {}
for pct in ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]:
    # pct_clean = pct.replace(".", "").replace("%", "")
    dot, df_result = show_precedence_graph(
        df,
        variants_variables=["backbone","frac_dtarget","tsk_pretext"],
        filters={
            "select_frac_dtarget": [pct],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TFC"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    tfc_dfs[f"df_finetune_tfc_rq4_{pct}"] = df_result
    summary = summarize_backbone_performance(df_result)
    display(summary)

lfr_dfs = {}
for pct in ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]:
    # pct_clean = pct.replace(".", "").replace("%", "")
    dot, df_result = show_precedence_graph(
        df,
        variants_variables=["backbone","frac_dtarget","tsk_pretext"],
        filters={
            "select_frac_dtarget": [pct],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["LFR"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    lfr_dfs[f"df_finetune_lfr_rq4_{pct}"] = df_result
    summary = summarize_backbone_performance(df_result)
    display(summary)

supervised_dfs = {}
for pct in ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]:
    # pct_clean = pct.replace(".", "").replace("%", "")
    dot, df_result = show_precedence_graph(
        df,
        variants_variables=["backbone","frac_dtarget","tsk_pretext"],
        filters={
            "select_frac_dtarget": [pct],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Supervised"],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    supervised_dfs[f"df_finetune_supervised_rq4_{pct}"] = df_result
    summary = summarize_backbone_performance(df_result)
    display(summary)



In [ ]:
df

In [ ]:
# # Inicializa lista para armazenar todos os summaries
# summary_rows = []

# # Defina os métodos e suas respectivas tabelas summary (que você já gerou anteriormente)
# methods = {
#     "TFC": list(tfc_dfs.values()),
#     "Diet": list(diet_dfs.values()),
#     "LFR": list(lfr_dfs.values()),
#     "Supervised": list(supervised_dfs.values())
# }

# fractions = [ "1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]

# # Itera sobre métodos e percentuais
# for method, summaries in methods.items():
#     for frac, summary_df in zip(fractions, summaries):
#         df_temp = summary_df.copy()
#         df_temp["Method"] = method
#         df_temp["Data Fraction"] = frac
#         summary_rows.append(df_temp)

# # Junta todos os summaries
# combined_df_rq4 = pd.concat(summary_rows, ignore_index=True)

# # Exibe o dataframe combinado
# display(combined_df_rq4)
# # Assuming you have these individual technique DataFrames:
# technique_dfs = [
#     combined_df_rq4
# ]

# technique_names = [
#     "TNC",
# ]

# # Generate all tables
# combined_df, technique_summary, backbone_totals = generate_performance_tables(
#     technique_dfs, 
#     technique_names
# )

# # Display results
# print("=== Technique-Specific Performance ===")
# display(technique_summary)

# print("\n=== Backbone Totals Across All Techniques ===")
# display(backbone_totals)


# # # Calcula vitórias/derrotas totais por backbone
# # summary_df_rq4 = combined_df_rq4.groupby("Backbone").agg({"Wins": "sum", "Losses": "sum"}).reset_index()
# # summary_df_rq4["Net Score"] = summary_df_rq4["Wins"] - summary_df_rq4["Losses"]
# # summary_df_rq4 = summary_df_rq4.sort_values("Net Score", ascending=False)

# # # Exibe resumo final
# # display(summary_df_rq4)


### P6* Qual o melhor backbone para HAR de acordo com o dataset?

Qual o melhor backbone para o TNC em cada dataset?
Qual o melhor backbone para o TFC em cada dataset?
Qual o melhor backbone para o Diet em cada dataset?
Qual o melhor backbone para o LFR em cada dataset?
Qual o melhor backbone para SL em cada dataset?


In [ ]:
df['d_target'].unique()

In [ ]:
# try:
#     base_df["frac_dtarget"] = base_df["frac_dtarget"].astype(float) * 100
#     base_df["frac_dtarget"] = base_df["frac_dtarget"].astype(str) + '%'
# except ValueError as e:
#     print(f"Error converting frac_dtarget: {e}")


base_df

In [ ]:
# full finetune by fraction
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]

results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
        # try:
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "frac_dtarget", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                "select_ft_strategy": ["Full Finetune"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset
        summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)
        # except Exception as e:
        #     print(f"Skipped {dataset}-{frac} due to: {e}")

# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)




In [ ]:
# full finetune by dataset
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    try:
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "frac_dtarget", "d_target"],
            filters={
                "select_d_target": [dataset],
                # "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
                "select_ft_strategy": ["Full Finetune"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset
        summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)
    except Exception as e:
        print(f"Skipped {dataset}-{frac} due to: {e}")

# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)


technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



# # Resumo por backbone (total geral)
# summary_df_rq4_by_dataset = (
#     combined_df_rq4_by_dataset
#     .groupby("Backbone")
#     .agg({"Wins": "sum", "Losses": "sum"})
#     .reset_index()
# )
# summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
# summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# # Exibe resumo final
# display(summary_df_rq4_by_dataset)


In [ ]:
base_df['ft_strategy'].unique()

In [ ]:
# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Freeze"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]

technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



In [ ]:
# with ts2vec 

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            # "select_tsk_pretext": ["TNC","Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# with ts2vec ssl freeze

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Freeze"],
            "select_tsk_pretext": ["TNC","Diet","TFC","LFR"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# Filter relevant columns including d_target
df_filtered = df[["backbone", "tsk_pretext", "metric", "d_target"]]

# Aggregate mean metric values per backbone, task, and d_target
df_grouped = df_filtered.groupby(["backbone", "tsk_pretext", "d_target"], as_index=False).mean()
df_grouped.rename(columns={"metric": "Score"}, inplace=True)

# Create a grid of plots
g = sns.FacetGrid(df_grouped, col="d_target", col_wrap=3, height=5, sharey=True)
g.map_dataframe(sns.barplot, x="tsk_pretext", y="Score", hue="backbone", palette=viridis_palette)
g.add_legend()
g.set_axis_labels("Task Pretext", "Accuracy")
g.set_titles("Target: {col_name}")

plt.tight_layout()
plt.show()

In [ ]:
# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["TFC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TFC",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# supervised cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["Supervised"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Supervised",
]

# Generate all tables
combined_df, technique_summary_supervised, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_supervised)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["LFR"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "LFR",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["TNC"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
# tfc cnn ts2vec across datasets and fractions

# with ts2vec partial

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
fractions = ["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"]


results_by_dataset = []

for dataset in datasets:
    for frac in fractions:
    
        dot, df_variant = show_precedence_graph(
            df,
            variants_variables=["backbone", "d_target"],
            filters={
                "select_d_target": [dataset],
                "select_frac_dtarget": [frac],
                # "select_backbones": ["ResNet-1D","CNN PF"],
                "select_ft_strategy": ["Full Finetune"],
                "select_tsk_pretext": ["Diet"],
                # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
            },
            show_stdev=True,
            apply_bonferroni=apply_correction_factor,
        )
        summary_df = summarize_backbone_performance(df_variant)
        summary_df["Dataset"] = dataset + " + " + frac
        summary_df["Backbone"] = summary_df["Backbone"] + " + " + frac
        # summary_df["Data Fraction"] = frac
        results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Diet",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)


In [ ]:
a

In [ ]:
display(technique_summary_supervised)
display(technique_summary_tfc)
display(technique_summary_lfr)
display(technique_summary_diet)
display(technique_summary_tnc)

## análise levy

Para pensar: poderíamos estipular algum critério de "suficiência". 

Por exemplo, considerar com que porcentagem de dados já é possível atingir 90% ou 95% do desempenho máx. (que seria o mínimo valor entre o supervised e o best SSL). 
A estabilidade trazida por SSL talvez fique evidente também nessa análise. 



In [ ]:
combined_df = pd.concat([
    technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_diet,
    technique_summary_tnc
])


combined_df = combined_df.drop_duplicates(subset=['Backbone','Technique'])
combined_df

In [ ]:

combined_df['Data Percentage'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[2].str.replace('.0', '').astype(int)
combined_df['Dataset Name'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[1]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df['Backbone'] = combined_df['Backbone'].str.split(r'\s+\+\s+').str[0]


combined_df

In [ ]:
combined_df.to_csv("rq4_full_finetune_results.csv", index=False)

In [ ]:
combined_df

In [ ]:
# Group by technique and find worst mean performance
worst_by_technique = combined_df.groupby('Technique')['Mean'].agg(['min', 'idxmin'])
worst_technique_results = combined_df.loc[worst_by_technique['idxmin']].sort_values('Mean')

display(worst_technique_results[['Technique', 'Dataset Name', 'Data Percentage', 'Backbone', 'Mean', 'Std']])

In [ ]:
# Group by data percentage and find worst mean performance
worst_by_percentage = combined_df.groupby('Data Percentage')['Mean'].agg(['min', 'idxmin'])
worst_percentage_results = combined_df.loc[worst_by_percentage['idxmin']].sort_values('Data Percentage')

display(worst_percentage_results[['Data Percentage', 'Dataset Name', 'Technique', 'Backbone', 'Mean', 'Std']])

In [ ]:
# Create a pivot table of worst performances
worst_pivot = combined_df.pivot_table(
    index=['Technique', 'Data Percentage'],
    values='Mean',
    aggfunc='min'
).nsmallest(50, 'Mean').reset_index()

# Merge back to get full details
worst_combined = worst_pivot.merge(
    combined_df,
    on=['Technique', 'Data Percentage', 'Mean'],
    how='left'
).drop_duplicates()

display(worst_combined[['Technique', 'Data Percentage', 'Dataset Name', 'Backbone', 'Mean', 'Std']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.boxplot(
    data=combined_df,
    x='Technique',
    y='Mean',
    hue='Data Percentage',
    palette='viridis'
)
plt.title('Performance Distribution by Technique and Data Percentage')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Get worst performance for each backbone-technique combination
worst_backbone_tech = combined_df.groupby(['Backbone', 'Technique'])['Mean'].min().reset_index()
worst_backbone_tech = worst_backbone_tech.merge(
    combined_df,
    on=['Backbone', 'Technique', 'Mean'],
    how='left'
).drop_duplicates()

print("Worst Performances by Backbone-Technique Pairs:")
display(worst_backbone_tech[['Backbone', 'Technique', 'Dataset Name', 'Data Percentage', 'Mean', 'Std']])

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(
    data=combined_df,
    x='Backbone',
    y='Mean',
    hue='Technique',
    palette='tab10'
)
plt.title('Performance Distribution by Backbone and Technique')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Pivot table for heatmap
heatmap_data = combined_df.pivot_table(
    index='Backbone',
    columns='Technique',
    values='Mean',
    aggfunc='min'
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    vmin=0,
    vmax=100
)
plt.title('Worst Performance Scores by Backbone and Technique')
plt.show()

In [ ]:
import plotly.express as px

fig = px.box(
    combined_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    points="all",
    hover_data=['Dataset Name', 'Data Percentage', 'Std'],
    title='Interactive Performance by Backbone and Technique'
)
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

In [ ]:
combined_df

In [ ]:
plot_df = combined_df.groupby(['Dataset Name', 'Backbone', 'Technique', 'Data Percentage'])['Mean'].agg(['mean','std']).reset_index()
plot_df.columns = ['Dataset', 'Backbone', 'Technique', 'Data_Percentage', 'Mean', 'Std']
plot_df

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Backbone',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Technique',  # Color grouping
    facet_col='Dataset',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Data_Percentage'
)
fig.show()

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create conditional highlight columns
plot_df['under_40'] = plot_df['Mean'] < 40
plot_df['under_20_percent'] = plot_df.groupby(['Dataset', 'Data_Percentage', 'Technique'])['Mean'].transform(
    lambda x: x < (x.max() * 0.8)
)

# Create main figure
fig = px.bar(
    plot_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    facet_col='Dataset',
    animation_frame='Data_Percentage',
    barmode='group',
    error_y='Std',
    color_discrete_map={
        'Supervised': '#636EFA',
        'TNC': '#EF553B',
        'LFR': '#00CC96'
    },
    width=1400,
    height=700,
    title='<b>Model Performance with Automatic Highlighting</b><br>'
          '<span style="font-size:12px">Red outline = Accuracy < 40% | '
          'Black hatch = 20% worse than best in group</span>'
)

# Add highlighting
for i, data in enumerate(fig.data):
    # Get corresponding dataframe subset
    subset = plot_df[
        (plot_df['Technique'] == data.name) & 
        (plot_df['Data_Percentage'] == fig.frames[0].name)
    ]
    
    # Add red outline for <40% accuracy
    fig.data[i].marker.line.color = [
        'red' if (x < 40) else 'rgba(0,0,0,0)' 
        for x in subset['Mean']
    ]
    fig.data[i].marker.line.width = 2
    
# Add reference lines and annotations
for col in range(1, len(plot_df['Dataset'].unique())+1):
    fig.add_hline(y=40, line_dash='dot', line_color='red', 
                 opacity=0.3, row=1, col=col)
    fig.add_annotation(
        text="",
        xref=f"x{col}", yref=f"y{col}",
        x=0.5, y=42, showarrow=False,
        font=dict(color='red', size=10)
    )

# Enhance layout
fig.update_layout(
    hoverlabel=dict(bgcolor='white', font_size=12),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    xaxis=dict(tickangle=45)
)

# Add dynamic title to animation frames
for frame in fig.frames:
    frame.layout.title = f'Data Percentage: {frame.name}'

fig.show()

In [ ]:
import plotly.express as px

# Create the figure with slower transitions
fig = px.bar(
    plot_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    facet_col='Dataset',
    animation_frame='Data_Percentage',
    barmode='group',
    error_y='Std',
    color_discrete_map={
        'Supervised': '#636EFA',
        'TNC': '#EF553B',
        'LFR': '#00CC96'
    },
    width=1400,
    height=700,
    title='<b>Model Performance with Automatic Highlighting</b><br>'
          '<span style="font-size:12px">Red outline = Accuracy < 40% | '
          'Black hatch = 20% worse than best in group</span>'
)

# Configure animation settings
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 2000  # 2 seconds per frame
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 1000  # 1 second transition
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['redraw'] = True  # Ensure smooth redraw

# Add highlighting (same as before)
for i, data in enumerate(fig.data):
    subset = plot_df[
        (plot_df['Technique'] == data.name) & 
        (plot_df['Data_Percentage'] == fig.frames[0].name)
    ]
    fig.data[i].marker.line.color = [
        'red' if (x < 40) else 'rgba(0,0,0,0)' 
        for x in subset['Mean']
    ]
    fig.data[i].marker.line.width = 2

# Add reference lines (same as before)
for col in range(1, len(plot_df['Dataset'].unique())+1):
    fig.add_hline(y=40, line_dash='dot', line_color='red', 
                 opacity=0.3, row=1, col=col)
    fig.add_annotation(
        text="40% Threshold",
        xref=f"x{col}", yref=f"y{col}",
        x=0.5, y=42, showarrow=False,
        font=dict(color='red', size=10)
    )

# Enhanced layout with slower easing
fig.update_layout(
    hoverlabel=dict(bgcolor='white', font_size=12),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    xaxis=dict(tickangle=45),
    transition={
        'duration': 1000,
        'easing': 'cubic-in-out'  # Smoother transition
    }
)

# Configure animation frames for smoothness
for frame in fig.frames:
    frame.layout.title = f'Data Percentage: {frame.name}'
    # Ensure all properties transition smoothly
    frame.layout.update(
        transition={'duration': 1000},
        xaxis={'autorange': True},
        yaxis={'autorange': True}
    )

# Add play/pause button with custom timing
fig.update_layout(
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 2000, "redraw": True}, 
                               "fromcurrent": True, 
                               "transition": {"duration": 1000, "easing": "cubic-in-out"}}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, 
                                "mode": "immediate",
                                "transition": {"duration": 0}}],
                "label": "❚❚ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 10},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 1.1,
        "yanchor": "top"
    }]
)

fig.show()

In [ ]:
import plotly.express as px

# Create the figure with slower transitions
fig = px.bar(
    plot_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    facet_col='Dataset',
    animation_frame='Data_Percentage',
    barmode='group',
    error_y='Std',
    color_discrete_map={
        'Supervised': '#636EFA',
        'TNC': '#EF553B',
        'LFR': '#00CC96'
    },
    width=1400,
    height=700,
    title='Model Performance based on backbone, technique, dataset and samples per class</b><br>LR 0,0001 at pretrain and downstream</b><br>'
)
fig.update_yaxes(range=[0, 100])
# Configure animation settings
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 3000  # 2 seconds per frame
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 0  # 1 second transition
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['redraw'] = True  # Ensure smooth redraw

# Add highlighting (same as before)
for i, data in enumerate(fig.data):
    subset = plot_df[
        (plot_df['Technique'] == data.name) & 
        (plot_df['Data_Percentage'] == fig.frames[0].name)
    ]
    fig.data[i].marker.line.color = [
        'red' if (x < 40) else 'rgba(0,0,0,0)' 
        for x in subset['Mean']
    ]
    fig.data[i].marker.line.width = 2

# Add reference lines (same as before)
for col in range(1, len(plot_df['Dataset'].unique())+1):
    fig.add_hline(y=100, line_dash='dot', line_color='black'),
    fig.add_hline(y=90, line_dash='dot', line_color='green', 
                 opacity=1, row=1, col=col)
    fig.add_hline(y=80, line_dash='dot', line_color='green', 
                 opacity=1, row=1, col=col)
    fig.add_hline(y=40, line_dash='dot', line_color='red', 
                 opacity=1, row=1, col=col)
    fig.add_hline(y=20, line_dash='dot', line_color='red', 
                 opacity=1, row=1, col=col)
    fig.add_annotation(
        text="",
        xref=f"x{col}", yref=f"y{col}",
        x=0.5, y=42, showarrow=False,
        font=dict(color='red', size=15)
    )

# Enhanced layout with slower easing
fig.update_layout(
    hoverlabel=dict(bgcolor='white', font_size=12),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.1,  # Moved legend below the plot
        xanchor='right',
        x=1
    ),
    xaxis=dict(tickangle=45),
    transition={
        'duration': 1000,
        'easing': 'cubic-in-out'  # Smoother transition
    },
    margin=dict(
        l=100,
        r=100,
        t=100,
        b=150  # Increased bottom margin for buttons
    )
)

# Configure animation frames for smoothness
for frame in fig.frames:
    frame.layout.title = f'Model Performance based on backbone, technique, dataset and samples per class</b><br> LR 0,0001 at pretrain and downstream, Samples Per Class: {frame.name}'
    # Ensure all properties transition smoothly
    frame.layout.update(
        transition={'duration': 1000},
        xaxis={'autorange': True},
        yaxis={'autorange': True}
    )

# Add play/pause button with custom timing
fig.update_layout(
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 2000, "redraw": True}, 
                               "fromcurrent": True, 
                               "transition": {"duration": 1000, "easing": "cubic-in-out"}}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, 
                                "mode": "immediate",
                                "transition": {"duration": 0}}],
                "label": "❚❚ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        # "pad": {"l": 20, "b": 10},  # Adjusted padding for bottom placement
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": -0.2,  # Moved buttons to bottom
        "yanchor": "bottom"
    }]
)

# Save to HTML with animation controls
fig.write_html(
    "experiment_random_samples.html",
    full_html=True,
    include_plotlyjs='cdn',
    config={
        'responsive': True,
        'displayModeBar': True,
        'scrollZoom': True,
        'modeBarButtonsToAdd': ['select2d', 'lasso2d']
    }
)
fig.show()

In [ ]:
this is the code

In [ ]:
a

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Technique',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Backbone',  # Color grouping
    facet_col='Dataset',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Data_Percentage'
)
fig.show()

In [ ]:
a

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Technique',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Backbone',  # Color grouping
    facet_col='Dataset',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Data_Percentage'
)
fig.show()

In [ ]:
fig = px.bar(
    plot_df,
    x='Backbone',
    y='Mean',
    color='Technique',
    facet_row='Data_Percentage',
    facet_col='Dataset',
    barmode='group',
    width=1400,
    height=2000
)
fig.show()

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Technique',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Backbone',  # Color grouping
    facet_col='Dataset',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Data_Percentage'
)
fig.show()

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Technique',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Backbone',  # Color grouping
    facet_col='Data_Percentage',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Dataset'
)
fig.show()

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Backbone',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Technique',  # Color grouping
    facet_col='Dataset',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Data_Percentage'
)
fig.show()

In [ ]:
fig = px.bar(
    data_frame=plot_df,
    x='Backbone',  # Or 'Dataset' or 'Backbone'
    y='Mean',
    color='Technique',  # Color grouping
    facet_col='Data_Percentage',  # Subplots by dataset
    barmode='group',  # 'stack' or 'relative'
    animation_frame='Dataset'
)
fig.show()

In [ ]:
worst_combinations = combined_df.nsmallest(10, 'Mean')[
    ['Backbone', 'Technique', 'Dataset Name', 'Data Percentage', 'Mean', 'Std']
].sort_values('Mean')

print("Top 10 Worst Backbone-Technique-Dataset Combinations:")
display(worst_combinations)

In [ ]:
# import pandas as pd

# # Pivot the table to compare Supervised vs. TFC
# comparison_df = combined_df.pivot_table(
#     index=['Dataset Name', 'Data Percentage'],
#     columns='Technique',
#     values='Mean',
#     aggfunc='first'
# ).reset_index()

# # Rename columns for clarity
# comparison_df.columns = ['Dataset', 'Data Percentage', 'Supervised', 'TFC']

# # Calculate the performance difference (TFC - Supervised)
# comparison_df['Performance_Diff'] = comparison_df['TFC'] - comparison_df['Supervised']
# comparison_df['Better_Method'] = comparison_df['Performance_Diff'].apply(
#     lambda x: 'TFC' if x > 0 else ('Supervised' if x < 0 else 'Tie')
# )
# comparison_df


In [ ]:
import pandas as pd

# 1. First find the best performance for each (dataset, data percentage) combination
def get_best_performance(df):
    return df.loc[df.groupby(['Dataset Name', 'Data Percentage', 'Technique'])['Mean'].idxmax()]

best_perf = get_best_performance(combined_df)
best_perf

In [ ]:
best_perf.to_csv("best_performance_by_dataset_and_percentage.csv", index=False)

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from matplotlib.lines import Line2D
# import re

# # Extract accuracy if needed
# def extract_accuracy(s):
#     return float(re.match(r"([0-9.]+)", s).group(1))

# # Process columns if not already done
# if 'SSL_Impact' in detailed_table.columns and detailed_table['SSL_Impact'].dtype == object:
#     detailed_table['SSL_Impact'] = detailed_table['SSL_Impact'].str.replace('+', '').astype(float)

# detailed_table['Data Percentage'] = detailed_table['Data Percentage'].astype(str).str.replace('%', '').astype(float)

# # Extract raw float accuracy - MODIFIED TO USE DIRECT VALUES
# detailed_table["SSL_Acc"] = detailed_table["Mean_TFC"]  # Use TFC mean directly
# detailed_table["Supervised_Acc"] = detailed_table["Mean_Supervised"]  # Use supervised mean directly

# # Plot styling
# sns.set_style("white")
# plt.rcParams.update({
#     'font.size': 14,
#     'axes.titlesize': 16,
#     'axes.labelsize': 13,
#     'xtick.labelsize': 13,
#     'ytick.labelsize': 13,
#     'legend.fontsize': 13
# })

# # Plot config
# datasets = detailed_table['Dataset Name'].unique()
# fig, axes = plt.subplots(3, 3, figsize=(14, 10))
# axes = axes.flatten()
# palette = sns.color_palette("Set2", 2)

# for i, dataset in enumerate(datasets):
#     ax = axes[i]
#     subset = detailed_table[detailed_table['Dataset Name'] == dataset].copy()
    
#     # Sort by data percentage to ensure proper line plotting
#     subset = subset.sort_values('Data Percentage')

#     # Plot supervised and SSL lines
#     ax.plot(subset['Data Percentage'], subset['Supervised_Acc'], color=palette[0], linestyle='--', linewidth=2.5)
#     ax.plot(subset['Data Percentage'], subset['SSL_Acc'], color=palette[1], linestyle='-', linewidth=2.5)

#     # Plot points
#     ax.scatter(subset['Data Percentage'], subset['Supervised_Acc'], color=palette[0], marker='o', s=100,
#                edgecolor='white', linewidth=1.2)
#     ax.scatter(subset['Data Percentage'], subset['SSL_Acc'], color=palette[1], marker='s', s=100,
#                edgecolor='white', linewidth=1.2)

#     # Plot SSL impact
#     for _, row in subset.iterrows():
#         ax.plot([row['Data Percentage'], row['Data Percentage']],
#                 [row['Supervised_Acc'], row['SSL_Acc']],
#                 color='gray', alpha=0.3, linestyle=':')
#         impact = f"{row['SSL_Impact']:+.1f}"
#         color = 'green' if row['SSL_Impact'] > 0 else 'red'
#         y_pos = max(row['Supervised_Acc'], row['SSL_Acc']) + 2
#         ax.text(row['Data Percentage'], y_pos, impact,
#                 ha='center', color=color, fontsize=12,
#                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

#     # ---------- Highlighting Key Points ----------
#     max_ssl = subset['SSL_Acc'].max()
#     best_row = subset[subset['SSL_Acc'] == max_ssl].iloc[0]
#     ax.scatter(best_row['Data Percentage'], best_row['SSL_Acc'], marker='*', s=300, color='gold', label='Max SSL')

#     # ≥90% of max
#     over_90 = subset[subset['SSL_Acc'] >= 0.90 * max_ssl]
#     if not over_90.empty:
#         min_row_90 = over_90.loc[over_90['Data Percentage'].idxmin()]
#         ax.scatter(min_row_90['Data Percentage'], min_row_90['SSL_Acc'],
#                    marker='^', s=180, color='blue', label='≥90% SSL')

#     # ≥95% of max
#     over_95 = subset[subset['SSL_Acc'] >= 0.95 * max_ssl]
#     if not over_95.empty:
#         min_row_95 = over_95.loc[over_95['Data Percentage'].idxmin()]
#         ax.scatter(min_row_95['Data Percentage'], min_row_95['SSL_Acc'],
#                    marker='v', s=180, color='purple', label='≥95% SSL')

#     # Formatting
#     ax.set_xscale('log')
#     ax.set_xticks([1, 5, 10, 50, 100])
#     ax.set_xticklabels(['1%', '5%', '10%', '50%', '100%'])
#     ax.set_xlabel('Data Percentage', fontweight='bold')
#     ax.set_ylabel('Balanced Accuracy (%)', fontweight='bold')
#     ax.set_title(f'{dataset}', fontweight='bold')
#     ax.set_ylim(0, 100)
#     ax.grid(False)

# # Hide empty subplots
# for i in range(len(datasets), len(axes)):
#     axes[i].axis('off')

# # Global legend - MODIFIED LEGEND LABELS
# custom_lines = [
#     Line2D([0], [0], color=palette[0], lw=3, linestyle='--', marker='o', label='Supervised'),
#     Line2D([0], [0], color=palette[1], lw=3, linestyle='-', marker='s', label='SSL (TFC)'),
#     Line2D([0], [0], color='gold', marker='*', lw=0, markersize=15, label='Max SSL'),
#     Line2D([0], [0], color='blue', marker='^', lw=0, markersize=10, label='≥90% of Max SSL'),
#     Line2D([0], [0], color='purple', marker='v', lw=0, markersize=10, label='≥95% of Max SSL'),
# ]
# fig.legend(handles=custom_lines, loc='upper center', ncol=3, frameon=False, fontsize=14)

# plt.tight_layout(pad=2.0)
# plt.subplots_adjust(top=0.88, bottom=0.1)
# plt.savefig('ssl_vs_supervised_with_thresholds.png', format='png', bbox_inches='tight', dpi=300)
# plt.show()

In [ ]:
# def create_detailed_comparison_table(df):
#     # First, verify the input data
#     print("Input data verification:")
#     print(df[['Dataset Name', 'Data Percentage', 'Technique', 'Mean']].head(10))
    
#     # Get best performance for each technique type
#     best_perfs = df.groupby(['Dataset Name', 'Data Percentage', 'Technique']).agg({
#         'Mean': 'max',
#         'Std': 'first',
#         'Backbone': 'first'
#     }).reset_index()
    
#     print("\nAfter grouping:")
#     print(best_perfs.head(10))
    
#     # Pivot to get supervised and TFC in separate columns
#     pivoted = best_perfs.pivot_table(
#         index=['Dataset Name', 'Data Percentage'],
#         columns='Technique',
#         values=['Mean', 'Std', 'Backbone'],
#         aggfunc='first'
#     ).reset_index()
    
#     # Flatten multi-index columns
#     pivoted.columns = ['_'.join(col).strip() for col in pivoted.columns.values]
    
#     # Calculate SSL impact (using TFC)
#     pivoted['SSL_Best_Mean'] = pivoted['Mean_TFC']
#     pivoted['SSL_Best_Technique'] = 'TFC'
#     pivoted['SSL_Impact'] = pivoted['SSL_Best_Mean'] - pivoted['Mean_Supervised']
    
#     # Format performance strings
#     def format_perf(row, technique):
#         if technique == 'Supervised':
#             mean = row['Mean_Supervised']
#             std = row['Std_Supervised']
#             backbone = row['Backbone_Supervised']
#             return f"{mean:.1f}±{std:.1f}({backbone})"
#         else:
#             mean = row['SSL_Best_Mean']
#             std = row['Std_TFC']
#             backbone = row['Backbone_TFC']
#             return f"{mean:.1f}±{std:.1f}({backbone},TFC)"
    
#     pivoted['Supervised_Perf'] = pivoted.apply(lambda x: format_perf(x, 'Supervised'), axis=1)
#     pivoted['SSL_Perf'] = pivoted.apply(lambda x: format_perf(x, 'SSL'), axis=1)
    
#     return pivoted.sort_values(['Dataset Name', 'Data Percentage'], ascending=[True, False])

# comparison_table = create_comparison_table(best_perf)
# display(comparison_table)

In [ ]:

# 2. Pivot to compare supervised vs SSL
def create_comparison_table(df):
    # Get best supervised
    supervised = df[df['Technique'] == 'Supervised'].copy()
    supervised['Best_Supervised'] = supervised['Mean'].astype(str) + '±' + supervised['Std'].astype(str) + \
                                  '(' + supervised['Backbone'] + ')'
    
    # Get best SSL (TFC or LFR)
    ssl = df[df['Technique'].isin(['TFC', 'LFR'])].copy()
    ssl['Best_SSL'] = ssl['Mean'].astype(str) + '±' + ssl['Std'].astype(str) + \
                     '(' + ssl['Backbone'] + ',' + ssl['Technique'] + ')'
    
    # Group and merge
    supervised_grouped = supervised.groupby(['Dataset Name', 'Data Percentage'])['Best_Supervised'].first().reset_index()
    ssl_grouped = ssl.groupby(['Dataset Name', 'Data Percentage']).agg({
        'Best_SSL': 'first',
        'Mean': 'max'
    }).reset_index()
    
    # Merge and calculate impact
    comparison = pd.merge(supervised_grouped, ssl_grouped, on=['Dataset Name', 'Data Percentage'])
    comparison['SSL_Impact'] = comparison['Mean'] - supervised_grouped.merge(
        ssl_grouped, on=['Dataset Name', 'Data Percentage']
    )['Mean']
    
    return comparison[['Dataset Name', 'Data Percentage', 'Best_Supervised', 'Best_SSL', 'SSL_Impact']]

# 3. Create and display the table
comparison_table = create_comparison_table(best_perf)
comparison_table = comparison_table.sort_values(['Dataset Name', 'Data Percentage'], ascending=[True, False])

# Format for better display
# comparison_table['Data Percentage'] = comparison_table['Data Percentage'].astype(str) + '%'
comparison_table['SSL_Impact'] = comparison_table['SSL_Impact'].apply(lambda x: f"+{x:.1f}" if x > 0 else f"{x:.1f}")

display(comparison_table)

In [ ]:
def create_detailed_comparison_table(df):
    # Get best performance for each technique type
    best_perfs = df.groupby(['Dataset Name', 'Data Percentage', 'Technique']).agg({
        'Mean': 'max',
        'Std': 'first',
        'Backbone': lambda x: x.iloc[0]  # Take first backbone when means are equal
    }).reset_index()
    
    # Pivot to get supervised, TFC and LFR in separate columns
    pivoted = best_perfs.pivot_table(
        index=['Dataset Name', 'Data Percentage'],
        columns='Technique',
        values=['Mean', 'Std', 'Backbone'],
        aggfunc='first'
    )
    
    # Flatten multi-index columns
    pivoted.columns = ['_'.join(col).strip() for col in pivoted.columns.values]
    pivoted = pivoted.reset_index()
    
    # Calculate SSL impact (using best of TFC or LFR)
    pivoted['SSL_Best_Mean'] = pivoted[['Mean_TFC', 'Mean_LFR']].max(axis=1)
    # pivoted['SSL_Best_Mean'] = pivoted[['Mean_TFC']].max(axis=1)
    pivoted['SSL_Best_Technique'] = pivoted.apply(
        lambda x: 'TFC' if x['Mean_TFC'] >= x['Mean_LFR'] else 'LFR',
        axis=1
    )
    # pivoted['SSL_Best_Technique'] = pivoted.apply(
    #     lambda x: 'TFC',
    #     axis=1
    # )
    pivoted['SSL_Impact'] = pivoted['SSL_Best_Mean'] - pivoted['Mean_Supervised']
    
    # Format performance strings - CORRECTED VERSION
    def format_perf(row, technique):
        if technique == 'Supervised':
            return f"{row['Mean_Supervised']:.1f}±{row['Std_Supervised']:.1f}({row['Backbone_Supervised']})"
        else:
            tech = row['SSL_Best_Technique']
            return f"{row['SSL_Best_Mean']:.1f}±{row[f'Std_{tech}']:.1f}({row[f'Backbone_{tech}']},{tech})"
    
    pivoted['Supervised_Perf'] = pivoted.apply(lambda x: format_perf(x, 'Supervised'), axis=1)
    pivoted['SSL_Perf'] = pivoted.apply(lambda x: format_perf(x, 'SSL'), axis=1)
    
    # Select and order columns
    # result = pivoted[[
    #     'Dataset Name', 'Data Percentage',
    #     'Mean_Supervised', 'Std_Supervised', 'Backbone_Supervised',
    #     'Mean_TFC', 'Std_TFC', 'Backbone_TFC',
    #     'Mean_LFR', 'Std_LFR', 'Backbone_LFR',
    #     'SSL_Best_Mean', 'SSL_Best_Technique', 'SSL_Impact',
    #     'Supervised_Perf', 'SSL_Perf'
    # ]]
    result = pivoted[[
        'Dataset Name', 'Data Percentage',
        'Mean_Supervised', 'Std_Supervised', 'Backbone_Supervised',
        'Mean_TFC', 'Std_TFC', 'Backbone_TFC',
        'SSL_Best_Mean', 'SSL_Best_Technique', 'SSL_Impact',
        'Supervised_Perf', 'SSL_Perf'
    ]]
    
    return result.sort_values(['Dataset Name', 'Data Percentage'], ascending=[True, False])

# Create and display the table
detailed_table = create_detailed_comparison_table(best_perf)

# Format for display
# detailed_table['Data Percentage'] = detailed_table['Data Percentage'].astype(str) + '%'
detailed_table['SSL_Impact'] = detailed_table['SSL_Impact'].apply(lambda x: f"+{x:.1f}" if x > 0 else f"{x:.1f}")

# Show only the most relevant columns for display
display_columns = [
    'Dataset Name', 'Data Percentage',
    'Supervised_Perf', 'SSL_Perf', 'SSL_Impact',
    'Mean_Supervised', 'Mean_TFC' # For verification
]
display(detailed_table[display_columns])

In [ ]:
display_columns = [
    'Dataset Name', 'Data Percentage',
    'Supervised_Perf', 'SSL_Perf', 'SSL_Impact', # For verification
]
display(detailed_table[display_columns])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D
import re

# Extract accuracy if needed
def extract_accuracy(s):
    return float(re.match(r"([0-9.]+)", s).group(1))

# Process columns
if 'SSL_Impact' in detailed_table.columns and detailed_table['SSL_Impact'].dtype == object:
    detailed_table['SSL_Impact'] = detailed_table['SSL_Impact'].str.replace('+', '').astype(float)

detailed_table['Data Percentage'] = detailed_table['Data Percentage'].astype(str).str.replace('%', '').astype(float)
detailed_table["SSL_Acc"] = detailed_table["SSL_Perf"].apply(extract_accuracy)
detailed_table["Supervised_Acc"] = detailed_table["Supervised_Perf"].apply(extract_accuracy)

# Plot styling
sns.set_style("white")
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13
})

# Ordered datasets
ordered_datasets = ['UCI', 'MS', 'KH', 'WISDM', 'RW-Waist', 'RW-Thigh']
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
palette = sns.color_palette("Set2", 2)

for i, dataset in enumerate(ordered_datasets):
    ax = axes[i]
    subset = detailed_table[detailed_table['Dataset Name'] == dataset].copy()

    # Plot supervised and SSL lines
    ax.plot(subset['Data Percentage'], subset['Supervised_Acc'], color=palette[1], linestyle='--', linewidth=2.5)
    ax.plot(subset['Data Percentage'], subset['SSL_Acc'], color=palette[0], linestyle='-', linewidth=2.5)

    # Plot points
    ax.scatter(subset['Data Percentage'], subset['Supervised_Acc'], color=palette[1], marker='o', s=80,
               edgecolor='white', linewidth=1.5, zorder=3)
    ax.scatter(subset['Data Percentage'], subset['SSL_Acc'], color=palette[0], marker='s', s=80,
               edgecolor='white', linewidth=1.5, zorder=3)

    # SSL impact lines and annotations
    for _, row in subset.iterrows():
        ax.plot([row['Data Percentage'], row['Data Percentage']],
                [row['Supervised_Acc'], row['SSL_Acc']],
                color='gray', alpha=0.3, linestyle=':')
        impact = f"{row['SSL_Impact']:+.1f}"
        color = 'green' if row['SSL_Impact'] > 0 else 'red'
        y_pos = max(row['Supervised_Acc'], row['SSL_Acc']) + 6  # <-- slightly higher
        ax.text(row['Data Percentage'], y_pos, impact,
                ha='center', color=color, fontsize=9,
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', boxstyle='round,pad=0.2'))

    # Highlight max overall
    subset['MaxOverall'] = subset[['SSL_Acc', 'Supervised_Acc']].max(axis=1)
    max_row = subset.loc[subset['MaxOverall'].idxmax()]
    ax.scatter(max_row['Data Percentage'], max_row['MaxOverall'], marker='*', s=250, color='gold', label='Max Performance', zorder=4)

    # ≥90% of max
    threshold_90 = 0.90 * max_row['MaxOverall']
    subset['Max_Acc_Per_Method'] = subset[['SSL_Acc']].max(axis=1)
    over_90 = subset[subset['Max_Acc_Per_Method'] >= threshold_90]
    if not over_90.empty:
        row_90 = over_90.loc[over_90['Data Percentage'].idxmin()]
        ax.scatter(row_90['Data Percentage'], row_90['Max_Acc_Per_Method'],
                   marker='v', s=80, color='blue', label='≥90% of Max', zorder=4)

    # ≥95% of max
    threshold_95 = 0.95 * max_row['MaxOverall']
    over_95 = subset[subset['Max_Acc_Per_Method'] >= threshold_95]
    if not over_95.empty:
        row_95 = over_95.loc[over_95['Data Percentage'].idxmin()]
        ax.scatter(row_95['Data Percentage'], row_95['Max_Acc_Per_Method'],
                   marker='^', s=80, color='purple', label='≥95% of Max', zorder=4)

    # Formatting
    ax.set_xscale('log')
    ax.set_xticks([1, 5,10,25, 50, 100,200,1000])
    ax.set_xticklabels(['1','5', '10','25', '50', '100', '200','max'])
    ax.set_yticks([20,30,40,50, 60, 70, 80, 90,100])
    ax.set_yticklabels(['20%','30%','40%','50%', '60%', '70%', '80%','90%', '100%'])
    ax.set_xlabel('Samples Per Class', fontweight='bold')

    ax.set_ylabel('Balanced Accuracy (%)', fontweight='bold')
    ax.set_title(f'{dataset}', fontweight='bold')
    ax.set_ylim(20, 100)
    ax.grid(False)

# Hide unused subplots
for i in range(len(ordered_datasets), len(axes)):
    axes[i].axis('off')

# Global legend
custom_lines = [
    Line2D([0], [0], color=palette[1], lw=3, linestyle='--', marker='o', label='Best Supervised'),
    Line2D([0], [0], color=palette[0], lw=3, linestyle='-', marker='s', label='Best SSL (TFC/LFR)'),
    Line2D([0], [0], color='gold', marker='*', lw=0, markersize=14, label='Max Performance'),
    Line2D([0], [0], color='blue', marker='v', lw=0, markersize=14, label='≥90% of Max'),
    Line2D([0], [0], color='purple', marker='^', lw=0, markersize=14, label='≥95% of Max'),
]
fig.legend(handles=custom_lines, loc='upper center', ncol=3, frameon=False, fontsize=14)

plt.tight_layout(pad=2.5)
plt.subplots_adjust(top=0.88, bottom=0.1)
plt.savefig('ssl_vs_supervised_max_overall.png', format='png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
detailed_table

In [ ]:
# Ensure accuracy columns are float
detailed_table['Supervised_Acc'] = detailed_table['Supervised_Acc'].astype(float)
detailed_table['SSL_Acc'] = detailed_table['SSL_Acc'].astype(float)

# Max performance per dataset across SSL and supervised
detailed_table['MaxOverall_Dataset'] = (
    detailed_table
    .groupby('Dataset Name')[['SSL_Acc', 'Supervised_Acc']]
    .transform('max')
    .max(axis=1)
)

# Compute thresholds
detailed_table['Threshold_90'] = 0.90 * detailed_table['MaxOverall_Dataset']
detailed_table['Threshold_95'] = 0.95 * detailed_table['MaxOverall_Dataset']

# Check if SSL crosses them
detailed_table['SSL_Over_90'] = detailed_table['SSL_Acc'] >= detailed_table['Threshold_90']
detailed_table['SSL_Over_95'] = detailed_table['SSL_Acc'] >= detailed_table['Threshold_95']


In [ ]:
detailed_table

In [ ]:
detailed_table.columns.to_list()

In [ ]:
display(detailed_table[['Dataset Name', 'Data Percentage', 'MaxOverall_Dataset', 'Mean_Supervised', 'SSL_Best_Mean', 'Threshold_90', 'Threshold_95', 'SSL_Over_90', 'SSL_Over_95']])

In [ ]:
# Filter for visual debugging
threshold_hits = detailed_table[detailed_table['SSL_Over_90'] | detailed_table['SSL_Over_95']]



In [ ]:
display(threshold_hits[['Dataset Name', 'Data Percentage', 'SSL_Acc', 'Threshold_90', 'Threshold_95', 'SSL_Over_90', 'SSL_Over_95']])

In [ ]:
threshold_hits


# mudar técnica ou backbone?

O que traz mais benefícios: mudar a estratégia de SSL dado um backbone e um dataset para pré-treino, ou mudar o backbone (mantendo a mesma técnica de SSL)? 

In [ ]:
# TNC

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TNC"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# TFC

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TFC"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TFC",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# Diet

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Diet"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Diet",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# LFR

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["LFR"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "LFR",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# supervised

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Supervised"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Supervised",
]

# Generate all tables
combined_df, technique_summary_supervised, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_supervised)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
combined_df = pd.concat([
    technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_tnc,
    technique_summary_diet,
])


# combined_df['Dataset Name'] = combined_df['Dataset'].str.split(r'\s+\+\s+').str[0]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df['Backbone'] = combined_df['Backbone'].str.extract(r'^(.+?)\s+\+')


combined_df

# analise completa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
import plotly.express as px

plt.style.use('seaborn')
sns.set_palette("husl")

def comprehensive_analysis(df):
    # 1. Filtro e preparação dos dados
    valid_techniques = ['Supervised', 'TFC', 'LFR', 'TNC', 'Diet']
    valid_backbones = ['CNN PF', 'TS Encoder', 'RNN', 'ResNet-1D', 'IMU Transformer']
    
    df = df.copy()
    df = df[df['Technique'].isin(valid_techniques)]
    df = df[df['Backbone'].isin(valid_backbones)]
    
    # 2. Análise de todas as combinações de técnicas
    tech_comparison = []
    
    for (dataset, backbone), group in df.groupby(['Dataset', 'Backbone']):
        tech_means = group.groupby('Technique')['Mean'].mean()
        if len(tech_means) >= 2:
            for tech1, tech2 in combinations(tech_means.index, 2):
                delta = tech_means[tech2] - tech_means[tech1]
                tech_comparison.append({
                    'Dataset': dataset,
                    'Backbone': backbone,
                    'Tech1': tech1,
                    'Tech2': tech2,
                    'Delta': delta,
                    'Tech1_Mean': tech_means[tech1],
                    'Tech2_Mean': tech_means[tech2],
                    'Comparison': f"{tech1} vs {tech2}"
                })
    
    tech_comp_df = pd.DataFrame(tech_comparison)
    
    # 3. Análise de todas as combinações de backbones
    bb_comparison = []
    
    for (dataset, technique), group in df.groupby(['Dataset', 'Technique']):
        bb_means = group.groupby('Backbone')['Mean'].mean()
        if len(bb_means) >= 2:
            for bb1, bb2 in combinations(bb_means.index, 2):
                delta = bb_means[bb2] - bb_means[bb1]
                bb_comparison.append({
                    'Dataset': dataset,
                    'Technique': technique,
                    'BB1': bb1,
                    'BB2': bb2,
                    'Delta': delta,
                    'BB1_Mean': bb_means[bb1],
                    'BB2_Mean': bb_means[bb2],
                    'Comparison': f"{bb1} vs {bb2}"
                })
    
    bb_comp_df = pd.DataFrame(bb_comparison)
    
    # 4. Cálculo de métricas agregadas
    tech_stats = tech_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count'])
    bb_stats = bb_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count'])
    
    # 5. Visualizações
    plt.figure(figsize=(18, 12))
    
    # Heatmap de comparação de técnicas
    plt.subplot(2, 2, 1)
    tech_pivot = tech_comp_df.pivot_table(index=['Backbone', 'Dataset'], 
                                         columns='Comparison', values='Delta')
    sns.heatmap(tech_pivot, cmap="coolwarm", center=0, annot=True, fmt=".1f")
    plt.title('Ganho de Performance entre Técnicas por Backbone/Dataset')
    plt.xticks(rotation=45, ha='right')
    
    # Distribuição dos ganhos por técnica
    plt.subplot(2, 2, 2)
    sns.boxplot(data=tech_comp_df, x='Delta', y='Comparison', showfliers=False)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('Distribuição dos Ganhos entre Técnicas')
    plt.xlabel('Ganho de Performance (%)')
    
    # Heatmap de comparação de backbones
    plt.subplot(2, 2, 3)
    bb_pivot = bb_comp_df.pivot_table(index=['Technique', 'Dataset'], 
                                     columns='Comparison', values='Delta')
    sns.heatmap(bb_pivot, cmap="coolwarm", center=0, annot=True, fmt=".1f")
    plt.title('Ganho de Performance entre Backbones por Técnica/Dataset')
    plt.xticks(rotation=45, ha='right')
    
    # Distribuição dos ganhos por backbone
    plt.subplot(2, 2, 4)
    sns.boxplot(data=bb_comp_df, x='Delta', y='Comparison', showfliers=False)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('Distribuição dos Ganhos entre Backbones')
    plt.xlabel('Ganho de Performance (%)')
    
    plt.tight_layout()
    plt.show()
    
    # 6. Análise estatística
    print("\n🔍 Análise Estatística das Comparações entre Técnicas:")
    print(tech_stats.sort_values('mean', ascending=False))
    
    print("\n🔍 Análise Estatística das Comparações entre Backbones:")
    print(bb_stats.sort_values('mean', ascending=False))
    
    # 7. Testes de significância
    print("\n📊 Testes de Significância:")
    
    # Para técnicas
    tech_pairs = tech_comp_df.groupby('Comparison')['Delta']
    for pair, values in tech_pairs:
        t_stat, p_val = stats.ttest_1samp(values, 0)
        print(f"{pair}: t={t_stat:.2f}, p={p_val:.4f} {'*' if p_val < 0.05 else ''}")
    
    # Para backbones
    bb_pairs = bb_comp_df.groupby('Comparison')['Delta']
    for pair, values in bb_pairs:
        t_stat, p_val = stats.ttest_1samp(values, 0)
        print(f"{pair}: t={t_stat:.2f}, p={p_val:.4f} {'*' if p_val < 0.05 else ''}")
    
    # 8. Visualização interativa (opcional - requer plotly)
    try:
        fig = px.sunburst(tech_comp_df, path=['Backbone', 'Comparison'], values='Delta',
                          color='Delta', color_continuous_scale='RdBu',
                          title='Impacto das Técnicas por Backbone')
        fig.show()
    except:
        print("\n⚠ Plotly não disponível para visualização interativa")
    
    return {
        'tech_comparisons': tech_comp_df,
        'bb_comparisons': bb_comp_df,
        'tech_stats': tech_stats,
        'bb_stats': bb_stats
    }

# Executar a análise completa
results = comprehensive_analysis(combined_df)

In [ ]:
tech_comp_df, bb_comp_df, tech_stats, bb_stats = (
    results['tech_comparisons'],
    results['bb_comparisons'],
    results['tech_stats'],
    results['bb_stats']
)


In [ ]:
tech_stats

In [ ]:
tech_stats = tech_stats.reset_index().rename(columns={'index': 'Comparison'})

# Function to process each row
def adjust_row(row):
    if row['mean'] < 0:
        a, b = row['Comparison'].split(' vs ')
        row['Comparison'] = f"{b} → {a}"
        row['mean'] = abs(row['mean'])
    else:
        row['Comparison'] = row['Comparison'].replace(' vs ', ' → ')
    return row

# Apply the transformation
tech_stats = tech_stats.apply(adjust_row, axis=1)
tech_stats = tech_stats.sort_values(by='mean',ascending=False)
tech_stats['mean'] = tech_stats['mean'].round(2)
tech_stats['std'] = tech_stats['std'].round(2)
display(tech_stats)

mean_of_means = tech_stats['mean'].mean().round(2)
mean_of_stds = tech_stats['std'].mean().round(2)

print(f"Mean of means: {mean_of_means}")
print(f"Mean of stds: {mean_of_stds}")


In [ ]:
bb_stats = bb_stats.reset_index().rename(columns={'index': 'Comparison'})

display(bb_stats)

# Apply the transformation
bb_stats = bb_stats.apply(adjust_row, axis=1)
bb_stats = bb_stats.sort_values(by='mean',ascending=False)
bb_stats['mean'] = bb_stats['mean'].round(2)
bb_stats['std'] = bb_stats['std'].round(2)
display(bb_stats)
mean_of_means = bb_stats['mean'].mean().round(2)
mean_of_stds = bb_stats['std'].mean().round(2)

print(f"Mean of means: {mean_of_means}")
print(f"Mean of stds: {mean_of_stds}")
